# Preparation: imports and data load

### imports

In [ ]:
# standard
import numpy as np
import matplotlib.pyplot as plt
import importlib
import os
import h5py
import json
from scipy.ndimage import gaussian_filter1d

import time

# custom

import axolotl_utils_ram
importlib.reload(axolotl_utils_ram)

from axolotl_utils_ram import extract_snippets_fast_ram, compute_and_plot_sta


import lighthouse_utils
importlib.reload(lighthouse_utils)

from lighthouse_utils import (
    find_valley_and_times,
    build_left_low_high_eis,
    filter_valley_spikes_by_batches,
    finalize_ei_with_cap,
    compute_prev_metrics, 
    plot_amp_vs_prev_metrics,
    plot_amp_prev_scatter_and_bar,
    lighthouse_morpho_pc_gmm
)


import plot_ei_waveforms
importlib.reload(plot_ei_waveforms)
import plot_ei_waveforms as pew

import collision_utils
importlib.reload(collision_utils)


import compute_sta_from_spikes
importlib.reload(compute_sta_from_spikes)


import joint_utils
importlib.reload(joint_utils)

from joint_utils import (
    cosine_two_eis
)

### load raw data and subtract baselines

In [ ]:
# --- Path and recording setup ---
# dat_path = "/Volumes/Lab/Users/alexth/raw_data/2018-08-07-9/data006.dat"
# dat_path = "/Volumes/Lab/Users/alexth/axolotl/201701261/data023.dat"
dat_path = "/Volumes/Lab/Users/alexth/axolotl/202510236/data018.dat"
n_channels = 512
dtype = np.int16

# --- Get total number of samples ---
file_size_bytes = os.path.getsize(dat_path)
total_samples = file_size_bytes // (np.dtype(dtype).itemsize * n_channels)

# --- Load entire file into RAM as int16 ---
raw_data = np.fromfile(dat_path, dtype=dtype, count=total_samples * n_channels)
raw_data = raw_data.reshape((total_samples, n_channels))  # shape: [T, C]


# ONLY EI POSITIONS
h5_in_path = '/Volumes/Lab/Users/alexth/axolotl/201703151_kilosort_data001_spike_times.h5'  # from MATLAB export, to get EI positions

with h5py.File(h5_in_path, 'r') as f:
    # Load electrode positions
    ei_positions = f['/ei_positions'][:].T  # shape becomes [512 x 2]



baseline_path = "/Volumes/Lab/Users/alexth/axolotl/202510236/data018_baseline_derivative_20k.json"

segment_len = 20_000
if os.path.exists(baseline_path):
    print(f"Loading baselines")
    with open(baseline_path, 'r') as f:
        data = json.load(f)
    baselines = np.array(data['baselines'], dtype=np.float32)
else:
    print(f"Computing baselines")
    baselines = axolotl_utils_ram.compute_baselines_int16_deriv_robust(raw_data, segment_len=segment_len, diff_thresh=10, trim_fraction=0.15) # shape (512, 360)

    with open(baseline_path, 'w') as f:
        json.dump({
            'baselines': baselines.tolist(),
        }, f)

print("subtracting baselines")

axolotl_utils_ram.subtract_segment_baselines_int16(raw_data=raw_data,
                                     baselines_f32=baselines,
                                     segment_len=segment_len) 

# triggers_mat_path='/Volumes/Lab/Users/alexth/axolotl/trigger_in_samples_201703151.mat'
# triggers_sec = loadmat(triggers_mat_path)['triggers'].flatten()


### load triggers

In [ ]:
# path = '/Volumes/Lab/Users/alexth/axolotl/201808079/data006_triggers.h5'
path = '/Volumes/Lab/Users/alexth/axolotl/201701261/data023_triggers.h5'
with h5py.File(path, 'r') as h5:
    triggers_sec = h5['/triggers'][...]         # shape (4303, 1) or (4303,)
triggers_sec = np.ravel(triggers_sec).astype(np.float64)  # 1D float64
len(triggers_sec)  # should be 4303

In [ ]:
# --- STREAMING LIGHTHOUSE LOOP: per-channel save, low RAM ---------------------
import time, os, gc, csv, json, gzip, pickle, datetime
from pathlib import Path

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from axolotl_utils_ram import extract_snippets_fast_ram


# -------- params ----------
chs = list(range(512))                 # or e.g. [222] / custom list
win = (-40, 80)
min_cosine = 0.75
# --- KMEANS CHECK PARAMS ---
KM_SUBSAMPLE_MAX   = 5_000   # subsample left_times to at most this many
KM_RMS_THRESH      = 5.0     # µV; select “significant” channels by EI RMS
KM_PC1_SINGLE_MAX  = 5.0     # PC1 EV% < 5  => single unit
KM_PC1_DOUBLE_MIN  = 7.0     # PC1 EV% > 7  => two units
KM_COS_THR         = 0.95    # if 5–7% band: cosine(EI0,EI1) > 0.95 => single
KM_LEAK_MAX        = 5       # if 5–7% & cosine<=: total leaks > 5 => single
KM_N_PC            = 10      # compute up to 10 PCs; we only use PC1/PC2
KM_RNG             = np.random.default_rng(123)


_step3_kwargs = dict(
    batch_size=100,
    reducer='median',
    tail_n=100,
    abs_cos_min=0.85,
    base_cos_cap=0.97,
    cos_slack=0.03,
    rms_gate=5.0,
    best_align_lag=3,
    use_abs=True,
)

# -------- output paths (timestamped run dir) ----------
# SAVE_DIR_ROOT = Path("/Volumes/Lab/Users/alexth/axolotl/201701261/LH_results")
SAVE_DIR_ROOT = Path("/Volumes/Lab/Users/alexth/axolotl/202510236/LH_results")
ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir = SAVE_DIR_ROOT / f"LH_run_{ts}"
run_dir.mkdir(parents=True, exist_ok=True)

# params for reproducibility
with open(run_dir / "params.json", "w") as f:
    json.dump({
        "chs": [int(x) for x in chs],
        "win": list(win),
        "min_cosine": float(min_cosine),
        "step3_kwargs": _step3_kwargs,
        "timestamp": ts
    }, f, indent=2)

# rolling summary CSV (append-safe)
summary_csv = run_dir / "summary.csv"
already_summarized = set()
if summary_csv.exists():
    with open(summary_csv, "r") as f:
        r = csv.reader(f)
        header = next(r, None)
        for row in r:
            if row:
                try: already_summarized.add(int(row[0]))
                except: pass
else:
    with open(summary_csv, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["ch","status",
                    "base_cosine","cos_low_mid","cos_mid_high",
                    "accepted_right_n","base_tail_cosine","dyn_floor",
                    "final_n","total_seconds"])

# rolling index files
lh_txt     = run_dir / "lighthouse_chans.txt"
nonlh_txt  = run_dir / "non_lighthouse_chans.txt"
s2fail_txt = run_dir / "step2_failed_chans.txt"
lh_set, nonlh_set, s2fail_set = set(), set(), set()

def _write_sets():
    lh_txt.write_text("\n".join(str(x) for x in sorted(lh_set)))
    nonlh_txt.write_text("\n".join(str(x) for x in sorted(nonlh_set)))
    s2fail_txt.write_text("\n".join(str(x) for x in sorted(s2fail_set)))

def _atomic_pickle_gz(obj, path: Path):
    tmp = path.with_suffix(path.suffix + ".tmp")
    with gzip.open(tmp, "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, path)  # atomic on POSIX

def _append_summary_row(res: dict):
    step2 = res.get("step2", {})
    step3 = res.get("step3", {})
    step4 = res.get("step4", {})
    left_n  = int(np.size(res.get("step1", {}).get("left_times", np.array([], dtype=np.int64))))
    right_n = int(np.size(step3.get("accepted_right_times", np.array([], dtype=np.int64))))
    left_plus_right_n = left_n + right_n

    row = [
        int(res.get("ch", -1)),
        res.get("status",""),
        float(step2.get("base_cosine", np.nan)),
        float(step2.get("cos_low_mid", np.nan)),
        float(step2.get("cos_mid_high", np.nan)),
        right_n,
        float(step3.get("base_tail_cosine", np.nan)),
        float(step3.get("dyn_floor", np.nan)),
        left_plus_right_n,   # <-- replaces the old "final_n" count
        float(res.get("timing", {}).get("total_s", np.nan)),
    ]

    with open(summary_csv, "a", newline="") as f:
        csv.writer(f).writerow(row)

def _kmeans_single_vs_double(raw_data, left_times, *, window, ch: int) -> dict:
    """
    Return a dict with K-means diagnostics and the decision:
      decision ∈ {'single_unit', 'two_units', 'skipped'}
      reason   ∈ {'pc1_ev<5', 'pc1_ev>7', 'cosine>thr', 'leaks>KM_LEAK_MAX',
                  'borderline_leaks<=KM_LEAK_MAX', 'too_few_snips', 'no_times'}
    Also returns: pc1_ev, pc2_ev, n_used, ksel, n0, n1, cosine, leak_total, leakA/B, indices, etc.
    """
    out = {
        "ch": int(ch), "decision": "skipped", "reason": "no_times",
        "pc1_ev": np.nan, "pc2_ev": np.nan, "n_used": 0, "ksel": 0,
        "n0": 0, "n1": 0, "cosine": np.nan,
        "leak_total": 0, "leakA_count": 0, "leakB_count": 0
    }

    lt = np.asarray(left_times, dtype=np.int64)
    if lt.size == 0:
        return out

    # subsample to cap cost
    if lt.size > KM_SUBSAMPLE_MAX:
        lt = lt[KM_RNG.choice(lt.size, KM_SUBSAMPLE_MAX, replace=False)]

    # extract ALL channels (explicit index array; None not allowed)
    all_ch = np.arange(raw_data.shape[1], dtype=np.int32)
    snips, valid_times = extract_snippets_fast_ram(
        raw_data, lt, window=window, selected_channels=all_ch
    )  # snips: [K, L, N]
    K, L, N = snips.shape
    if N < 10:
        out.update(decision="skipped", reason="too_few_snips")
        return out

    # EI mean for channel selection
    ei_mean = snips.mean(axis=2).astype(np.float32)      # [K, L]
    rms = np.sqrt(np.mean(ei_mean**2, axis=1))           # [K]
    sel = np.flatnonzero(rms > KM_RMS_THRESH)
    if sel.size == 0:
        sel = np.argsort(rms)[-16:]
        sel.sort()
    Ksel = int(sel.size)

    # build features: concat selected channels (no z-score, no DC)
    X = snips[sel, :, :].transpose(2, 0, 1).reshape(N, Ksel * L).astype(np.float32)

    # PCA → PCs
    n_pc = int(min(KM_N_PC, X.shape[0], X.shape[1]))
    pca = PCA(n_components=n_pc, svd_solver='randomized', random_state=0)
    Xpc = pca.fit_transform(X)
    vr = pca.explained_variance_ratio_
    pc1_ev = float(vr[0] * 100.0)
    pc2_ev = float(vr[1] * 100.0) if n_pc >= 2 else 0.0

    # KMeans k=2 in PC space
    km = KMeans(n_clusters=2, n_init=50, random_state=0)
    lab = km.fit_predict(Xpc)
    idx0 = np.flatnonzero(lab == 0); n0 = int(idx0.size)
    idx1 = np.flatnonzero(lab == 1); n1 = int(idx1.size)

    out.update(pc1_ev=pc1_ev, pc2_ev=pc2_ev, n_used=int(N), ksel=Ksel, n0=n0, n1=n1,
               idx0=idx0, idx1=idx1, times0=valid_times[idx0], times1=valid_times[idx1],
               selected_channels=sel.tolist())

    # --- Decision cascade ---
    if pc1_ev < KM_PC1_SINGLE_MAX:
        out.update(decision="single_unit", reason="pc1_ev<5")
        return out
    if pc1_ev > KM_PC1_DOUBLE_MIN:
        out.update(decision="two_units", reason="pc1_ev>7")
        return out

    # borderline 5–7%: compare EIs by cosine (abs)
    ei0 = snips[:, :, idx0].mean(axis=2).astype(np.float32) if n0 > 0 else np.zeros((K, L), np.float32)
    ei1 = snips[:, :, idx1].mean(axis=2).astype(np.float32) if n1 > 0 else np.zeros((K, L), np.float32)
    v0 = ei0[sel].ravel(); v1 = ei1[sel].ravel()
    denom = (np.linalg.norm(v0) * np.linalg.norm(v1)) + 1e-12
    cos = float(np.abs(np.dot(v0, v1)) / denom)
    out["cosine"] = cos

    if cos > KM_COS_THR:
        out.update(decision="single_unit", reason="cosine>thr")
        return out

    # cosine <= thr → leaks test on PC1–PC2
    P2 = Xpc[:, :2].astype(np.float32)
    A = P2[idx0]; B = P2[idx1]
    if n0 >= 2 and n1 >= 2:
        # nearest within-cluster (exclude self via n_neighbors=2 → take second)
        dA_self = NearestNeighbors(n_neighbors=2).fit(A).kneighbors(A, return_distance=True)[0][:, 1]
        dB_self = NearestNeighbors(n_neighbors=2).fit(B).kneighbors(B, return_distance=True)[0][:, 1]
        # nearest cross-cluster
        dA_cross = NearestNeighbors(n_neighbors=1).fit(B).kneighbors(A, return_distance=True)[0][:, 0]
        dB_cross = NearestNeighbors(n_neighbors=1).fit(A).kneighbors(B, return_distance=True)[0][:, 0]
        leakA = (dA_cross < dA_self)
        leakB = (dB_cross < dB_self)
        leak_total = int(leakA.sum() + leakB.sum())
        out.update(leak_total=leak_total,
                   leakA_count=int(leakA.sum()), leakB_count=int(leakB.sum()))
        if leak_total > KM_LEAK_MAX:
            out.update(decision="single_unit", reason="leaks>KM_LEAK_MAX")
        else:
            out.update(decision="two_units", reason="borderline_leaks<=KM_LEAK_MAX")
    else:
        # not enough to evaluate leaks — be conservative and call two units on cosine split
        out.update(decision="two_units", reason="borderline_too_few_for_leaks")

    return out


def _run_one_channel(ch: int):
    """Run the 4-step pipeline for a single channel and return a result dict."""
    out = dict(ch=int(ch), status=None, timing={})
    t0 = time.time()

    # Step 1
    try:
        step1 = find_valley_and_times(
            raw_data, ch,
            window=win, start=0, stop=None,
            bin_width=10.0, valley_bins=5,
            min_valid_count=500,
            ratio_base=3, ratio_step=200, ratio_floor=2, ratio_cap=10
        )
        out["step1"] = step1
        out["timing"]["step1_s"] = time.time() - t0
        if not bool(step1.get("accepted", False)):
            out["status"] = "step1_failed"
            return out
        
        # --- Step 1.5: K-means single-vs-double check on left_times only ---
        t1k = time.time()
        try:
            kmres = _kmeans_single_vs_double(
                raw_data, step1["left_times"], window=win, ch=ch
            )
            out["kmeans"] = kmres
            out["timing"]["kmeans_s"] = time.time() - t1k
        except Exception as e:
            out["kmeans"] = {"decision": "skipped", "reason": f"error:{type(e).__name__}", "error": str(e)}
            out["timing"]["kmeans_s"] = time.time() - t1k

    except Exception as e:
        out["status"] = "error_step1"
        out["error"]  = f"{type(e).__name__}: {e}"
        return out

    # Step 2
    t1 = time.time()
    try:
        step2 = build_left_low_high_eis(
            raw_data, step1["left_times"], step1["left_vals"],
            window=win, left_cap=500, reducer='median', rms_gate=5.0
        )
        out["step2"] = step2
        out["timing"]["step2_s"] = time.time() - t1
        base_cos = float(step2.get("base_cosine", np.nan))
        if not np.isfinite(base_cos) or base_cos < min_cosine:
            out["status"] = "step2_failed"
            return out
    except Exception as e:
        out["status"] = "error_step2"
        out["error"]  = f"{type(e).__name__}: {e}"
        return out

    # Step 3
    t2 = time.time()
    try:
        step3 = filter_valley_spikes_by_batches(
            raw_data,
            step1["rightk_times_by_amp"],
            step1["left_times"], step1["left_vals"],
            window=win,
            **_step3_kwargs
        )
        out["step3"] = step3
        out["timing"]["step3_s"] = time.time() - t2
    except Exception as e:
        out["status"] = "error_step3"
        out["error"]  = f"{type(e).__name__}: {e}"
        return out

    # Step 4
    t3 = time.time()
    try:
        step4 = finalize_ei_with_cap(
            raw_data,
            step1["left_times"],
            step3.get("accepted_right_times", np.array([], dtype=np.int64)),
            window=win,
            final_max_spikes=5_000,
            reducer='median',
            rng_seed=123,
            ref_ch = ch
        )
        out["step4"] = step4
        out["timing"]["step4_s"] = time.time() - t3
        out["timing"]["total_s"] = time.time() - t0
        out["status"] = "ok"
    except Exception as e:
        out["status"] = "error_step4"
        out["error"]  = f"{type(e).__name__}: {e}"

    return out

# -------- main loop: save each channel immediately & free memory --------------
for ch in chs:
    ch = int(ch)
    ch_file = run_dir / f"LH_ch_{ch:03d}.pkl.gz"

    # run
    print(f"[ch {ch:>3}] START compute", flush=True)
    t_comp0 = time.time()

    res = _run_one_channel(ch)

    # DROP the massive blob before pickling
    if 'step4' in res and isinstance(res['step4'], dict):
        res['step4'].pop('final_snips', None)


    t_comp1 = time.time()
    print(f"[ch {ch:>3}] compute done in {t_comp1 - t_comp0:.1f}s; START save", flush=True)


    # save per-channel immediately (compressed)
    t_save0 = time.time()
    try:
        _atomic_pickle_gz(res, ch_file)
        t_save1 = time.time()
        print(f"[ch {ch:>3}] save done in {t_save1 - t_save0:.1f}s", flush=True)
    except Exception as e:
        print(f"[ch {ch:>3}] ERROR saving after {time.time() - t_save0:.1f}s: {e}", flush=True)


    # update rolling indices
    status = res.get("status", "")
    if status == "step1_failed" or status.startswith("error_"):
        nonlh_set.add(ch)
    else:
        lh_set.add(ch)
    if status == "step2_failed":
        s2fail_set.add(ch)
    _write_sets()

    # append a summary row (idempotent w.r.t. duplicates via tracking set)
    if ch not in already_summarized:
        _append_summary_row(res)
        already_summarized.add(ch)

    # progress
    msg = f"[ch {ch:>3}] {status}"
    if status == "ok":
        n_right = int(np.size(res["step3"].get("accepted_right_times", np.array([], dtype=np.int64))))
        left_n  = int(np.size(res["step1"].get("left_times", np.array([], dtype=np.int64))))
        used_n  = left_n + n_right
        msg += f" | base_cos={res['step2']['base_cosine']:.3f} | right_ok={n_right} | final_n={used_n} | {res['timing']['total_s']:.1f}s"

    print(msg)

    # free memory eagerly
    del res
    gc.collect()

print("\nRun dir:", run_dir)
print("Index files updating live:",
      lh_txt.name, nonlh_txt.name, s2fail_txt.name, "and", summary_csv.name)
# -----------------------------------------------------------------------------


# --- Optional helpers: quick loading of indices later -------------------------
# Example: lighthouse_chans = np.loadtxt(run_dir/'lighthouse_chans.txt', dtype=int)
#          non_lighthouses = np.loadtxt(run_dir/'non_lighthouse_chans.txt', dtype=int)


In [ ]:
from pathlib import Path
import numpy as np

# root = Path("/Volumes/Lab/Users/alexth/axolotl/201701261/LH_results")
root = Path("/Volumes/Lab/Users/alexth/axolotl/202510236/LH_results")
run_dir = sorted(root.glob("LH_run_*"))[-1]  # latest run

# load the text file we saved incrementally
lh_file = run_dir / "lighthouse_chans.txt"
lighthouse_chans = np.loadtxt(lh_file, dtype=int) if lh_file.exists() else np.array([], int)

print("Lighthouse channels:", lighthouse_chans.tolist())
print("Total:", lighthouse_chans.size)


In [ ]:
import gzip, pickle, json
import numpy as np
from pathlib import Path

def _np_to_py(x):
    """Make numpy types JSON-safe."""
    if isinstance(x, (np.generic,)):
        return x.item()
    if isinstance(x, np.ndarray):
        return x.tolist()
    raise TypeError(f"Type not JSON-serializable: {type(x)}")

def get_kmeans_info(run_dir: Path, ch: int, save_json: bool = False):
    """
    Load and return the full kmeans dict for a channel.
    Raises FileNotFoundError or KeyError with a clear message if missing.
    """
    p = run_dir / f"LH_ch_{int(ch):03d}.pkl.gz"
    if not p.exists():
        raise FileNotFoundError(f"No file for ch {ch}: {p}")

    with gzip.open(p, "rb") as f:
        res = pickle.load(f)

    km = res.get("kmeans", None)
    if km is None:
        raise KeyError(f"'kmeans' missing for ch {ch} (status={res.get('status')})")

    # quick on-screen summary
    summary = {
        "ch": int(ch),
        "decision": km.get("decision"),
        "reason": km.get("reason"),
        "pc1_ev": km.get("pc1_ev"),
        "pc2_ev": km.get("pc2_ev"),
        "cosine": km.get("cosine"),
        "leak_total": km.get("leak_total"),
        "n0": km.get("n0"), "n1": km.get("n1"),
        "n_used": km.get("n_used"), "ksel": km.get("ksel"),
    }
    print("KMEANS summary:", summary)

    if save_json:
        out_json = run_dir / f"kmeans_ch_{int(ch):03d}.json"
        with open(out_json, "w") as f:
            json.dump(km, f, indent=2, default=_np_to_py)
        print(f"Saved full kmeans dict to: {out_json}")

    return km

# ---- Example usage ----
ch = 6
km = get_kmeans_info(run_dir, ch, save_json=False)
km


In [ ]:
import gzip, pickle, glob, re
from pathlib import Path

def print_lighthouse_single_units(run_dir: Path):
    paths = sorted(glob.glob(str(run_dir / "LH_ch_*.pkl.gz")))
    singles = []

    for p in paths:
        name = Path(p).name
        m = re.match(r"LH_ch_(\d+)\.pkl\.gz$", name)
        if not m:
            continue
        ch = int(m.group(1))

        with gzip.open(p, "rb") as f:
            res = pickle.load(f)

        # Lighthouse = step1 accepted (fallback: status not step1_failed/error_ if step1 missing)
        step1_ok = bool(res.get("step1", {}).get("accepted", False))
        if not step1_ok:
            status = str(res.get("status", ""))
            if status == "step1_failed" or status.startswith("error_"):
                continue
            step1_ok = True  # treat as lighthouse if not explicitly failed

        km = res.get("kmeans")
        if step1_ok and km and km.get("decision") == "single_unit":
            singles.append(ch)

    singles = sorted(set(singles))
    print(f"lighthouse single-unit channels ({len(singles)}):")
    if singles:
        print(singles)
    else:
        print("[]")

# SAVE_DIR_ROOT = Path("/Volumes/Lab/Users/alexth/axolotl/201701261/LH_results/LH_run_20251105_195338/")
# run_dir = SAVE_DIR_ROOT


# --- usage ---
print_lighthouse_single_units(run_dir)


In [ ]:
import gzip, pickle, glob, re
from pathlib import Path
from typing import Optional
from collections import defaultdict


# ---- helpers ----

def _load_res(path: Path):
    with gzip.open(path, "rb") as f:
        return pickle.load(f)

def _get_ch_from_name(name: str) -> Optional[int]:

    m = re.match(r"LH_ch_(\d+)\.pkl\.gz$", name)
    return int(m.group(1)) if m else None

def _extract_final_ei(step4: dict):
    # Try typical keys, fallback raises
    for k in ("final_ei", "ei", "ei_final", "EI", "finalEI"):
        if k in step4 and isinstance(step4[k], np.ndarray):
            ei = np.asarray(step4[k], dtype=np.float32)
            if ei.ndim == 2:
                return ei
    raise KeyError("Could not find final EI in step4 dict (looked for final_ei/ei/ei_final/EI/finalEI).")

def _rms_by_channel(ei: np.ndarray) -> np.ndarray:
    # ei: [C, L]
    return np.sqrt(np.mean(ei**2, axis=1)).astype(np.float32)

def _topk_channels(ei: np.ndarray, k: int = 3) -> np.ndarray:
    rms = _rms_by_channel(ei)
    k = min(k, rms.size)
    idx = np.argpartition(rms, -k)[-k:]
    return np.sort(idx)

def _cosine_two_eis(ei0: np.ndarray, ei1: np.ndarray, *, rms_gate: float = 5.0,
                    use_abs: bool = True, best_align_lag: int = 6):
    """
    Cosine similarity between two EIs with per-channel RMS mask and best lag in [-L,+L].
    Returns: dict(cosine, best_lag, nch_used, overlap_len)
    """
    ei0 = np.asarray(ei0, dtype=np.float32)
    ei1 = np.asarray(ei1, dtype=np.float32)
    C = min(ei0.shape[0], ei1.shape[0])
    L0, L1 = ei0.shape[1], ei1.shape[1]
    best = {"cosine": np.nan, "best_lag": 0, "nch_used": 0, "overlap_len": 0}

    def _score_at_lag(lag):
        # align along time, crop to overlap
        if lag >= 0:
            L = min(L0 - lag, L1)
            if L <= 3:  # require a few samples
                return None
            X = ei0[:C, lag:lag+L]
            Y = ei1[:C, :L]
        else:
            L = min(L0, L1 + lag)
            if L <= 3:
                return None
            X = ei0[:C, :L]
            Y = ei1[:C, -lag:-lag+L]

        # channel mask by RMS
        rms0 = np.sqrt(np.mean(X**2, axis=1))
        rms1 = np.sqrt(np.mean(Y**2, axis=1))
        m = (rms0 > rms_gate) | (rms1 > rms_gate)
        if not np.any(m):
            return None

        x = X[m].ravel()
        y = Y[m].ravel()
        denom = (np.linalg.norm(x) * np.linalg.norm(y)) + 1e-12
        if denom == 0.0:
            return None
        cos = float(np.dot(x, y) / denom)
        if use_abs:
            cos = abs(cos)
        return {"cosine": cos, "best_lag": lag, "nch_used": int(m.sum()), "overlap_len": int(L)}

    for lag in range(-best_align_lag, best_align_lag + 1):
        s = _score_at_lag(lag)
        if s is None:
            continue
        if not np.isfinite(best["cosine"]) or s["cosine"] > best["cosine"]:
            best = s
    return best

# ---- 1) collect lighthouse singles & their EIs ----

pkls = sorted(glob.glob(str(run_dir / "LH_ch_*.pkl.gz")))
singles = []
for p in pkls:
    ch = _get_ch_from_name(Path(p).name)
    if ch is None: 
        continue
    res = _load_res(Path(p))
    step1_ok = bool(res.get("step1", {}).get("accepted", False))
    km = res.get("kmeans", None)
    if not step1_ok or not km or km.get("decision") != "single_unit":
        continue
    step4 = res.get("step4", {})
    if not step4:
        continue
    try:
        ei = _extract_final_ei(step4)  # [C, L]
    except Exception as e:
        print(f"[skip ch {ch}] no final EI: {e}")
        continue
    rms = _rms_by_channel(ei)
    main_ch = int(np.argmax(rms))
    top3 = set(_topk_channels(ei, k=3).tolist())
        # --- spike counts (total = left + accepted_right) ---
    step1 = res.get("step1", {})
    step3 = res.get("step3", {})
    n_left  = int(np.size(step1.get("left_times", np.array([], dtype=np.int64))))
    n_right = int(np.size(step3.get("accepted_right_times", np.array([], dtype=np.int64))))
    n_total = n_left + n_right

    singles.append({
        "ch": ch,
        "ei": ei,
        "rms": rms,
        "main_ch": main_ch,
        "top3": top3,
        "n_spikes": n_total,   # <-- added
    })


print(f"Found {len(singles)} lighthouse singles with final EIs.")

# ---- 2) pairwise cosine on overlapping-top3 pairs ----

pairs = []
for i in range(len(singles)):
    for j in range(i+1, len(singles)):
        si, sj = singles[i], singles[j]
        overlap = si["top3"].intersection(sj["top3"])
        if not overlap:
            continue
        sim = _cosine_two_eis(si["ei"], sj["ei"], rms_gate=5.0, use_abs=True, best_align_lag=6)
        pairs.append({
            "ch_i": si["ch"], "main_i": si["main_ch"], "N_i": int(singles[i]["n_spikes"]),
            "ch_j": sj["ch"], "main_j": sj["main_ch"], "N_j": int(singles[j]["n_spikes"]),
            "overlap_ch": sorted(list(overlap)),
            "cosine": sim["cosine"],
            "best_lag": sim["best_lag"],
            "nch_used": sim["nch_used"],
            "overlap_len": sim["overlap_len"],
        })


# sort by cosine descending
pairs_sorted = sorted(pairs, key=lambda r: (-(r["cosine"] if np.isfinite(r["cosine"]) else -np.inf),
                                            -r["nch_used"]))

print(f"Compared {len(pairs_sorted)} suspect pairs (top-3 overlap >= 1).")
if not pairs_sorted:
    print("No overlapping-top3 pairs to report.")
else:
    # print all pairs; tweak format as needed
    for k, r in enumerate(pairs_sorted, 1):
        print(f"[{k:03d}] ch{r['ch_i']:3d} (main {r['main_i']:3d}, N={r['N_i']})  "
              f"<-> ch{r['ch_j']:3d} (main {r['main_j']:3d}, N={r['N_j']})  "
              f"| overlap {r['overlap_ch']}  "
              f"| cosine={r['cosine']:.4f}  lag={r['best_lag']:>+2d}  "
              f"nch_used={r['nch_used']:3d}  T={r['overlap_len']:3d}")

# ---------- DEDUPLICATE LIGHTHOUSE SINGLES BY EI COSINE ----------

DUP_COS_THR = 0.95  # duplicates if cosine >= 0.95

# map channels <-> indices in `singles`
ch_list = [s["ch"] for s in singles]
ch_to_idx = {ch: i for i, ch in enumerate(ch_list)}
n = len(singles)

# Union-Find (disjoint-set) to merge duplicate-connected components
parent = list(range(n))
def _find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i
def _union(i, j):
    ri, rj = _find(i), _find(j)
    if ri != rj:
        parent[rj] = ri

# add edges for duplicate pairs (cosine >= threshold)
dup_edges = []
for r in pairs_sorted:
    cos = r.get("cosine", np.nan)
    if np.isfinite(cos) and cos >= DUP_COS_THR:
        i = ch_to_idx[r["ch_i"]]
        j = ch_to_idx[r["ch_j"]]
        _union(i, j)
        dup_edges.append((r["ch_i"], r["ch_j"], float(cos)))

# collect components
groups = defaultdict(list)
for i in range(n):
    groups[_find(i)].append(i)

# choose representative per component:
#   1) max n_spikes
#   2) tie-break by peak RMS across channels
#   3) tie-break by smaller channel id
def _rep_key(i):
    n_spk = int(singles[i].get("n_spikes", 0))
    peak = float(np.max(singles[i].get("rms", np.zeros(1, dtype=np.float32))))
    ch    = int(singles[i]["ch"])
    return (n_spk, peak, -ch)  # max() picks higher n_spk, higher peak, then smaller ch

unique_idxs = []
dropped_idxs = []
for root, nodes in groups.items():
    if len(nodes) == 1:
        unique_idxs.append(nodes[0])
        continue
    rep = max(nodes, key=_rep_key)
    unique_idxs.append(rep)
    for i in nodes:
        if i != rep:
            dropped_idxs.append(i)

unique_channels = sorted(int(singles[i]["ch"]) for i in unique_idxs)

print(f"\nDuplicate threshold: cosine >= {DUP_COS_THR:.2f}")
print(f"Found {len(dup_edges)} duplicate edges among {len(singles)} lighthouse singles.")
print(f"Unique lighthouse channels ({len(unique_channels)}):")
print(unique_channels)

# (optional) show which ones were dropped from each duplicate group
if dropped_idxs:
    print("\nDedup groups (kept -> dropped):")
    for root, nodes in groups.items():
        if len(nodes) <= 1:
            continue
        rep = max(nodes, key=_rep_key)
        keep_ch = int(singles[rep]["ch"])
        drop_ch = sorted(int(singles[i]["ch"]) for i in nodes if i != rep)
        print(f"  keep ch{keep_ch}: drop {drop_ch}")



In [ ]:
print(unique_channels)
print(len(unique_channels))


In [ ]:
unique_channels[69]

### plot all lighthouse units for inspection

In [ ]:
# --- helpers for per-channel "print+plot → PDF" capture ---
import os, gc, io, contextlib, warnings, traceback
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import textwrap

# minimal FigureCollector: collects every Matplotlib figure born while active
class FigureCollector:
    def __enter__(self):
        import matplotlib.pyplot as _plt
        self._orig_show = _plt.show
        self._pre_ids = set(_plt.get_fignums())
        _plt.show = lambda *a, **k: None  # no-op so figs aren’t closed by show()
        return self
    def __exit__(self, exc_type, exc, tb):
        import matplotlib.pyplot as _plt
        post = set(_plt.get_fignums())
        self.figures = [_plt.figure(fid) for fid in sorted(post - self._pre_ids)]
        _plt.show = self._orig_show
        return False  # don’t swallow exceptions

# capture stdout/stderr/warnings -> text
class LogCollector:
    def __init__(self, capture_stderr=True, capture_warnings=True):
        self._buf = io.StringIO()
        self._redir_out = contextlib.redirect_stdout(self._buf)
        self._redir_err = contextlib.redirect_stderr(self._buf) if capture_stderr else contextlib.nullcontext()
        self._warn_cm = warnings.catch_warnings(record=True) if capture_warnings else contextlib.nullcontext()
        self._warn_records = None
        self.text = ""
    def __enter__(self):
        self._redir_out.__enter__(); self._redir_err.__enter__()
        if isinstance(self._warn_cm, contextlib.AbstractContextManager):
            self._warn_records = self._warn_cm.__enter__(); warnings.simplefilter("always")
        return self
    def __exit__(self, exc_type, exc, tb):
        if self._warn_records is not None:
            for w in self._warn_records:
                try: self._buf.write(f"[warning] {w.category.__name__}: {w.message}\n")
                except Exception: pass
            self._warn_cm.__exit__(exc_type, exc, tb)
        self._redir_err.__exit__(exc_type, exc, tb)
        self._redir_out.__exit__(exc_type, exc, tb)
        self.text = self._buf.getvalue()
        return False

# turn a long text blob into one or more “LOG” pages (figures)
def logs_to_figures(text, title=None, max_chars=110, max_lines=55, figsize=(11, 8.5), fontsize=9):
    s = "(no output)" if (text is None or not str(text).strip()) else str(text).replace("\r\n","\n").replace("\r","\n")
    lines = []
    for raw in s.split("\n"):
        wrapped = textwrap.wrap(raw, width=max_chars, replace_whitespace=False, drop_whitespace=False) or [""]
        lines.extend(wrapped)
    figs = []
    for i in range(0, len(lines), max_lines):
        page = "\n".join(lines[i:i+max_lines])
        fig = plt.figure(figsize=figsize)
        if title: 
            try: fig.suptitle(title, y=0.98)
            except Exception: pass
        fig.text(0.02, 0.96, page, va="top", ha="left", family="monospace", fontsize=fontsize)
        figs.append(fig)
    return figs


In [ ]:

# --- params ---
SUBSAMPLE_MAX = 5_000
WINDOW = (-40, 80)          # same window you use in LH step1
RMS_THRESH = 5.0            # µV threshold on EI RMS
N_PC = 10                   # use up to 10 PCs for k-means; plot PC1 vs PC2
RNG = np.random.RandomState(123)
win = (-40, 80)
min_cosine = 0.75
sample_rate_hz = 20_000   # 20 kHz
bin_width_ms   = 0.5
max_ms         = 300.0
dt_ms          = 1000
sigma_ms       = 2500
OUT_DIR = "/Volumes/Lab/Users/alexth/axolotl/201701261/LH_results/plots"
os.makedirs(OUT_DIR, exist_ok=True)


# ---- single-page composition helpers ----
import io, math, textwrap
from matplotlib.backends.backend_pdf import PdfPages

def fig_to_image(fig, dpi=450):
    """Render a Matplotlib Figure to a high-DPI PNG for crisp thumbnails."""
    buf = io.BytesIO()
    fig.savefig(
        buf, format="png", dpi=dpi,
        bbox_inches="tight", facecolor="white", edgecolor="white"
    )
    buf.seek(0)
    img = plt.imread(buf)
    buf.close()
    return img


def summarize_log(text, tail_lines=28, max_chars=6000):
    """Compact the printout: keep only the last N lines (fits on one page)."""
    s = "" if text is None else str(text)
    s = s[-max_chars:]
    lines = s.splitlines()
    if len(lines) > tail_lines:
        keep = lines[-tail_lines:]
        keep.insert(0, f"[log truncated] showing last {tail_lines} of {len(lines)} lines")
        return "\n".join(keep)
    return s

def compose_channel_page(figs, log_text, ch, *, page_size=(11, 8.5), cols=2, dpi=120, max_tiles=6):
    """
    Build ONE Matplotlib figure that contains:
      - a compact log panel at the top
      - up to `max_tiles` thumbnails of collected figures (rasterized)
    """
    # 1) convert figures → images
    imgs_all = [fig_to_image(f, dpi=dpi) for f in figs]
    extra = max(0, len(imgs_all) - max_tiles)
    imgs = imgs_all[:max_tiles]
    n = len(imgs)

    # 2) layout: log row + image grid
    rows = max(1, math.ceil(max(1, n) / cols))
    log_ratio = 0.28 if n >= 4 else 0.35

    page = plt.figure(figsize=page_size)
    gs = page.add_gridspec(nrows=rows + 1, ncols=cols,
                           height_ratios=[log_ratio] + [1] * rows,
                           wspace=0.02, hspace=0.02)

    # top: logs
    ax0 = page.add_subplot(gs[0, :]); ax0.axis("off")
    title = f"Channel {int(ch)}  |  {len(figs)} plots captured"
    ax0.text(0.0, 1.0, title, transform=ax0.transAxes,
             fontsize=11, fontweight="bold", va="top", ha="left")
    ax0.text(0.0, 0.96, summarize_log(log_text), transform=ax0.transAxes,
             fontsize=7, family="monospace", va="top", ha="left")

    # grid: plot thumbnails
    if n == 0:
        ax = page.add_subplot(gs[1, 0]); ax.axis("off")
        ax.text(0.5, 0.5, "(no plots captured)", ha="center", va="center")
    else:
        for i, img in enumerate(imgs):
            r, c = divmod(i, cols)
            ax = page.add_subplot(gs[1 + r, c]); ax.axis("off")
            ax.imshow(img)

    # overflow note
    if extra > 0:
        ax = page.add_subplot(gs[-1, -1]); ax.axis("off")
        ax.text(0.5, 0.5, f"+ {extra} more not shown", ha="center", va="center", fontsize=10)

    return page



for ch in unique_channels[69:137]:
    pdf_path = os.path.join(OUT_DIR, f"LH_ch{int(ch):03d}.pdf")
    with FigureCollector() as _FC, LogCollector(capture_stderr=True, capture_warnings=True) as _LC:
        try:
            print(f"[CHANNEL {ch}]")

            step1 = find_valley_and_times(
                raw_data, ch,
                window=win, start=0, stop=None,   # use the SAME window here as you’ll use later
                bin_width=10.0, valley_bins=5,
                min_valid_count=500,
                ratio_base=3, ratio_step=500, ratio_floor=2, ratio_cap=10  # required ratio clipped at 10×
            )
            print("valley [%.1f, %.1f) Left=%d Valley=%d R*=%g"
                % (step1['valley_low'], step1['valley_high'], step1['left_count'], step1['valley_count'], step1['required_ratio']))
            print(f"Left spikes for filter: {len(step1['left_times'])}, right spikes: {len(step1['rightk_times_by_amp'])}")
            assert step1['accepted'], "valley acceptance failed"

            plt.figure(figsize=(12,3))
            plt.bar(step1['amp_hist_edges'][:-1],
                    step1['amp_hist_counts'],
                    width=np.diff(step1['amp_hist_edges']),
                    align='edge')
            plt.axvspan(step1['valley_low'], step1['valley_high'], color='r', alpha=0.3)
            plt.title(f"Channel {ch}: amplitude histogram with valley")
            plt.ylim(0, 5000)
            plt.show()


            # --- inputs ---
            left_times = np.asarray(step1['left_times'], dtype=np.int64)

            # 1) subsample spike times (uniform, no replacement)
            n_total = left_times.size
            if n_total == 0:
                raise ValueError("No spikes in step1['left_times'].")
            pick = left_times if n_total <= SUBSAMPLE_MAX else left_times[RNG.choice(n_total, SUBSAMPLE_MAX, replace=False)]

            # 2) extract snippets on ALL channels
            # snips: [K (channels), L (samples), N (spikes)]
            snips, valid_times = extract_snippets_fast_ram(
                raw_data, pick, window=WINDOW, selected_channels=np.arange(512, dtype=np.int32)
            )
            # print("Completed snippets]")

            K, L, N = snips.shape
            # print(f"snips shape = {snips.shape}; kept {N}/{pick.size} after window bounds")

            if N < 10:
                raise ValueError("Too few valid snippets after extraction; adjust window or subsample size.")

            # 3) EI by mean over spikes (no DC removal)
            ei_mean = snips.mean(axis=2).astype(np.float32)   # [K, L]

            # 4) select significant channels by EI RMS
            rms = np.sqrt(np.mean(ei_mean**2, axis=1))        # [K]
            sel = np.flatnonzero(rms > RMS_THRESH)

            # fallback if nothing passes the gate: take top-16 by RMS so we can still cluster
            if sel.size == 0:
                sel = np.argsort(rms)[-16:]
                sel.sort()
                print(f"[warn] No channels over RMS>{RMS_THRESH}. Using top-16 by RMS instead.")
            Ksel = sel.size
            # print(f"Selected {Ksel} channels by RMS>{RMS_THRESH} µV.")

            # 5) build per-spike feature by concatenating waveforms across selected channels
            # shape -> [N, Ksel*L]
            X = snips[sel, :, :].transpose(2, 0, 1).reshape(N, Ksel * L).astype(np.float32)

            # 6) PCA (no z-scoring, no centering beyond PCA’s internal centering)
            n_pc = int(min(N_PC, X.shape[0], X.shape[1]))
            pca = PCA(n_components=n_pc, svd_solver='randomized', random_state=0)
            Xpc = pca.fit_transform(X)       # [N, n_pc]
            vr = pca.explained_variance_ratio_

            # 7) k-means (k=2) in PC space
            km = KMeans(n_clusters=2, n_init=50, random_state=0)
            lab = km.fit_predict(Xpc)        # [N]

            ei_c0 = snips[:,:,lab == 0].mean(axis=2).astype(np.float32)   # [K, L]
            ei_c1 = snips[:,:,lab == 1].mean(axis=2).astype(np.float32)   # [K, L]
            base_cos, base_nch, base_lag, base_olen = cosine_two_eis(
                ei_c0, ei_c1, rms_gate=5.0, use_abs=True, best_align_lag=3
            )
            # 8) scatter PC1 vs PC2
            plt.figure(figsize=(4, 4))
            plt.scatter(Xpc[:, 0], Xpc[:, 1], c=lab, s=10, alpha=0.85)
            # mark cluster centers (in PC1/PC2 projection)
            c0 = Xpc[lab == 0, :2].mean(axis=0)
            c1 = Xpc[lab == 1, :2].mean(axis=0)
            plt.scatter([c0[0], c1[0]], [c0[1], c1[1]], marker='X', s=140, edgecolor='k', linewidths=1)

            plt.xlabel(f"PC1 ({vr[0]*100:.1f}% var)")
            plt.ylabel(f"PC2 ({vr[1]*100:.1f}% var)" if n_pc >= 2 else "PC2")
            plt.title(f"k=2 on PCs from concat({Ksel}ch × {L}t); N={N}; cosine {base_cos:.2f}")
            plt.grid(alpha=0.3)
            plt.tight_layout()
            plt.show()



            step2 = build_left_low_high_eis(
                raw_data, step1['left_times'], step1['left_vals'],
                window=win, left_cap=500, reducer='median', rms_gate=5.0  # k_each = min(500, N_left_valid//2) per end
            )
            print("base cosine (low vs high) =", round(step2['base_cosine'], 3), "(high vs mid) =", round(step2['cos_mid_high'], 3), "(low vs mid) =", round(step2['cos_low_mid'], 3))
            assert step2['base_cosine'] >= min_cosine, "left_low vs left_high too different; abort"



            # Run tail-based batch filter on valley/right set:
            step3 = filter_valley_spikes_by_batches(
                raw_data,
                step1['rightk_times_by_amp'],  # valley + fill to >=500, amplitude-sorted
                step1['left_times'], step1['left_vals'],
                window=(-40, 80),
                batch_size=100,
                reducer='median',
                tail_n=100,
                abs_cos_min=0.85,
                base_cos_cap=0.97,
                cos_slack=0.02,
                rms_gate=5.0,
                best_align_lag=3,
                use_abs=True
            )

            print("base tail cosine:", round(step3['base_tail_cosine'],3), "dyn_floor:", round(step3['dyn_floor'],3))
            print("batch cosines:", [f"{x:.2f}" for x in step3['batch_cosines']])
            print("accepted right spikes:", step3['accepted_right_times'].size)


            step4 = finalize_ei_with_cap(
                raw_data,
                step1['left_times'],
                step3['accepted_right_times'],
                window=win,
                final_max_spikes=5_000,   # set None to use ALL left+accepted valley spikes
                reducer='median',
                rng_seed=123,
                ref_ch=ch
            )
            print("Union size =", step4['final_times'].size, "main ch ", step4['main_ch'])

            ######### PLOTS #######
            # EI

            fig = plt.figure(figsize=(20, 8))
            ax = fig.add_subplot(111)
            pew.plot_ei_waveforms(
                step4['final_ei'], positions=ei_positions, scale=70.0, ax=ax, colors=["black", "red"], 
                alpha=1.0, linewidth=1.0, box_height=1.0, box_width=50.0, aspect=0.5,
            )
            ax.set_title('Final EI')
            plt.show()

            spikes_in_samples = np.sort(step4['final_times'])  # ensure ascending

            # STA and time course
            sta0, red_tc0, green_tc0, blue_tc0 = compute_and_plot_sta(spikes_in_samples, triggers_sec, STA_DEPTH=30, STA_OFFSET=2, STA_CHUNK=1000, STA_REFRESH=2, SEED=22222, W=20, H=40, label="all", peak_frame=None, mode="bw")


            # --- ISI histogram ---
            t_ms = (spikes_in_samples / float(sample_rate_hz)) * 1000.0  # Spikes in milliseconds
            isi_ms = np.diff(t_ms)
            bins = np.arange(0.0, max_ms + bin_width_ms, bin_width_ms)
            counts, edges = np.histogram(isi_ms, bins=bins)
            centers = edges[:-1] + 0.5 * bin_width_ms


            # --- Smoothed firing rate ---
            dt = dt_ms / 1000
            sigma_samples = sigma_ms / dt_ms
            spikes_sec = (spikes_in_samples / float(sample_rate_hz))
            total_duration = spikes_sec.max() + 0.1 if len(spikes_sec) > 0 else 1.0
            time_vector = np.arange(0, total_duration, dt)
            rate_counts, _ = np.histogram(spikes_sec, bins=np.append(time_vector, total_duration))
            rate = gaussian_filter1d(rate_counts / dt, sigma=sigma_samples)


            # --- Combined figure ---
            fig, axes = plt.subplots(1, 2, figsize=(20, 3))

            # Left: ISI histogram
            axes[0].plot(centers, counts, lw=1.5)
            axes[0].set_xlim(0, max_ms)
            axes[0].set_xlabel("ISI (ms)")
            axes[0].set_ylabel("Count")
            axes[0].set_title(f"ISI histogram (bin={bin_width_ms} ms, max={max_ms} ms)\nN pairs={isi_ms.size}, Unit 1")

            # Right: Smoothed firing rate
            axes[1].plot(time_vector, rate, color='red')
            axes[1].set_xlim(0, total_duration)
            axes[1].set_xlabel("Time (s)")
            axes[1].set_ylabel("Hz")
            axes[1].set_title("Smoothed Firing Rate")

            plt.tight_layout()
            plt.show()


            # high/low spikes 
            amps        = step4['final_amplitudes']   # positive trough magnitudes on main channel
            window_ms = 50.0 # ms

            # Compute features (no plotting):
            m = compute_prev_metrics(step4['final_times'], sample_rate_hz=sample_rate_hz, window_ms=window_ms)
            # print("Median prev-ISI (ms):", np.nanmedian(m['isi_prev_ms']),
            #     " | median prev-count:", np.median(m['prev_count']))

            plot_amp_prev_scatter_and_bar(
                step4['final_times'], amps,
                sample_rate_hz=sample_rate_hz,
                window_ms=window_ms,
                max_k=20,            # show k = 0..20 (20+ collapsed into the last bar)
                min_count=5,        # only show bars with ≥20 spikes
                error='std',
                alpha=0.15,
                title_suffix=f"ch {ch}, window={window_ms} ms"
            )
            plt.show()

        except Exception:
            # still emit an ERROR page if this channel blows up
            tb = traceback.format_exc()
            fig = plt.figure(figsize=(11, 8.5))
            fig.suptitle(f"Ch {int(ch)} · ERROR", y=0.98)
            fig.text(0.01, 0.95, tb, va="top", ha="left", family="monospace", fontsize=9)
            # keep the figure alive; it will be saved below

    # add a “LOG” page and write the PDF
    # compose ONE summary page (plots + compact logs) and save exactly one page
    _summary = compose_channel_page(
        _FC.figures or [], _LC.text, ch,
        page_size=(14, 10.5),   # larger page → larger tiles
        cols=2,                 # one column → bigger thumbnails
        dpi=450,                # matches fig_to_image default
        max_tiles=6             # keep the biggest 3 plots; shows “+N more” if overflow
    )

    with PdfPages(pdf_path) as _pdf:
        _pdf.savefig(_summary, bbox_inches="tight")
    plt.close(_summary)

    # close originals to free memory
    for _f in _FC.figures or []:
        try: plt.close(_f)
        except Exception: pass
    plt.close('all'); gc.collect()
    print(f"[saved one-page] {pdf_path}")



# Find lighthouse units

In [ ]:
ch = 7

win = (-40, 80)

min_cosine = 0.75

start_time = time.time()

step1 = find_valley_and_times(
    raw_data, ch,
    window=win, start=0, stop=None,   # use the SAME window here as you’ll use later
    bin_width=10.0, valley_bins=5,
    min_valid_count=500,
    ratio_base=3, ratio_step=500, ratio_floor=2, ratio_cap=10  # required ratio clipped at 10×
)
print("valley [%.1f, %.1f) Left=%d Valley=%d R*=%g"
      % (step1['valley_low'], step1['valley_high'], step1['left_count'], step1['valley_count'], step1['required_ratio']))
print(f"Left spikes for filter: {len(step1['left_times'])}, right spikes: {len(step1['rightk_times_by_amp'])}")
assert step1['accepted'], "valley acceptance failed"

plt.figure(figsize=(12,3))
plt.bar(step1['amp_hist_edges'][:-1],
        step1['amp_hist_counts'],
        width=np.diff(step1['amp_hist_edges']),
        align='edge')
plt.axvspan(step1['valley_low'], step1['valley_high'], color='r', alpha=0.3)
plt.title(f"Channel {ch}: amplitude histogram with valley")
plt.ylim(0, 5000)
plt.show()

valley_time = time.time()
elapsed = valley_time - start_time 
print(f"Valley time: {elapsed:.1f} seconds.\n")

step2 = build_left_low_high_eis(
    raw_data, step1['left_times'], step1['left_vals'],
    window=win, left_cap=500, reducer='median', rms_gate=5.0  # k_each = min(500, N_left_valid//2) per end
)
print("base cosine (low vs high) =", round(step2['base_cosine'], 3))
print("base cosine (high vs mid) =", round(step2['cos_mid_high'], 3))
print("base cosine (low vs mid) =", round(step2['cos_low_mid'], 3))
assert step2['base_cosine'] >= min_cosine, "left_low vs left_high too different; abort"

ei_time = time.time()
elapsed = ei_time - valley_time 
print(f"EI time: {elapsed:.1f} seconds.\n")

# Run tail-based batch filter on valley/right set:
step3 = filter_valley_spikes_by_batches(
    raw_data,
    step1['rightk_times_by_amp'],  # valley + fill to >=500, amplitude-sorted
    step1['left_times'], step1['left_vals'],
    window=(-40, 80),
    batch_size=100,
    reducer='median',
    tail_n=100,
    abs_cos_min=0.85,
    base_cos_cap=0.97,
    cos_slack=0.02,
    rms_gate=5.0,
    best_align_lag=3,
    use_abs=True
)

print("base tail cosine:", round(step3['base_tail_cosine'],3), "dyn_floor:", round(step3['dyn_floor'],3))
print("batch cosines:", step3['batch_cosines'])
print("accepted right spikes:", step3['accepted_right_times'].size)

filter_time = time.time()
elapsed = filter_time - ei_time 
print(f"Filter time: {elapsed:.1f} seconds.\n")


step4 = finalize_ei_with_cap(
    raw_data,
    step1['left_times'],
    step3['accepted_right_times'],
    window=win,
    final_max_spikes=5_000,   # set None to use ALL left+accepted valley spikes
    reducer='median',
    rng_seed=123,
    ref_ch=ch
)
print("final EI built from", step4['final_times_used'].size, "snippets; union size =", step4['final_times'].size, "main ch ", step4['main_ch'])

final_ei_time = time.time()
elapsed = final_ei_time - filter_time 
print(f"Final EI time: {elapsed:.1f} seconds.")



In [ ]:
plt.figure(figsize=(12,3))
plt.bar(step1['amp_hist_edges'][:-1],
        step1['amp_hist_counts'],
        width=np.diff(step1['amp_hist_edges']),
        align='edge')
plt.axvspan(step1['valley_low'], step1['valley_high'], color='r', alpha=0.3)
plt.title(f"Channel {ch}: amplitude histogram with valley")
plt.ylim(0, 15000)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from axolotl_utils_ram import extract_snippets_fast_ram


# ch = 234
win = (-40, 80)

min_cosine = 0.75

start_time = time.time()

step1 = find_valley_and_times(
    raw_data, ch,
    window=win, start=0, stop=None,   # use the SAME window here as you’ll use later
    bin_width=10.0, valley_bins=5,
    min_valid_count=500,
    ratio_base=3, ratio_step=500, ratio_floor=2, ratio_cap=10  # required ratio clipped at 10×
)
print("valley [%.1f, %.1f) Left=%d Valley=%d R*=%g"
      % (step1['valley_low'], step1['valley_high'], step1['left_count'], step1['valley_count'], step1['required_ratio']))
print(f"Left spikes for filter: {len(step1['left_times'])}, right spikes: {len(step1['rightk_times_by_amp'])}")
assert step1['accepted'], "valley acceptance failed"

plt.figure(figsize=(12,3))
plt.bar(step1['amp_hist_edges'][:-1],
        step1['amp_hist_counts'],
        width=np.diff(step1['amp_hist_edges']),
        align='edge')
plt.axvspan(step1['valley_low'], step1['valley_high'], color='r', alpha=0.3)
plt.title(f"Channel {ch}: amplitude histogram with valley")
plt.ylim(0, 15000)
plt.show()


# --- params ---
SUBSAMPLE_MAX = 5_000
WINDOW = (-40, 80)          # same window you use in LH step1
RMS_THRESH = 5.0            # µV threshold on EI RMS
N_PC = 10                   # use up to 10 PCs for k-means; plot PC1 vs PC2
RNG = np.random.RandomState(123)

# --- inputs ---
left_times = np.asarray(step1['left_times'], dtype=np.int64)

# 1) subsample spike times (uniform, no replacement)
n_total = left_times.size
if n_total == 0:
    raise ValueError("No spikes in step1['left_times'].")
pick = left_times if n_total <= SUBSAMPLE_MAX else left_times[RNG.choice(n_total, SUBSAMPLE_MAX, replace=False)]

# 2) extract snippets on ALL channels
# snips: [K (channels), L (samples), N (spikes)]
snips, valid_times = extract_snippets_fast_ram(
    raw_data, pick, window=WINDOW, selected_channels=np.arange(512, dtype=np.int32)
)
print("Completed snippets]")

K, L, N = snips.shape
print(f"snips shape = {snips.shape}; kept {N}/{pick.size} after window bounds")

if N < 10:
    raise ValueError("Too few valid snippets after extraction; adjust window or subsample size.")

# 3) EI by mean over spikes (no DC removal)
ei_mean = snips.mean(axis=2).astype(np.float32)   # [K, L]

# 4) select significant channels by EI RMS
rms = np.sqrt(np.mean(ei_mean**2, axis=1))        # [K]
sel = np.flatnonzero(rms > RMS_THRESH)

# fallback if nothing passes the gate: take top-16 by RMS so we can still cluster
if sel.size == 0:
    sel = np.argsort(rms)[-16:]
    sel.sort()
    print(f"[warn] No channels over RMS>{RMS_THRESH}. Using top-16 by RMS instead.")
Ksel = sel.size
print(f"Selected {Ksel} channels by RMS>{RMS_THRESH} µV.")

# 5) build per-spike feature by concatenating waveforms across selected channels
# shape -> [N, Ksel*L]
X = snips[sel, :, :].transpose(2, 0, 1).reshape(N, Ksel * L).astype(np.float32)

# 6) PCA (no z-scoring, no centering beyond PCA’s internal centering)
n_pc = int(min(N_PC, X.shape[0], X.shape[1]))
pca = PCA(n_components=n_pc, svd_solver='randomized', random_state=0)
Xpc = pca.fit_transform(X)       # [N, n_pc]
vr = pca.explained_variance_ratio_

# 7) k-means (k=2) in PC space
km = KMeans(n_clusters=2, n_init=50, random_state=0)
lab = km.fit_predict(Xpc)        # [N]

# 8) scatter PC1 vs PC2
plt.figure(figsize=(6, 6))
plt.scatter(Xpc[:, 0], Xpc[:, 1], c=lab, s=10, alpha=0.85)
# mark cluster centers (in PC1/PC2 projection)
c0 = Xpc[lab == 0, :2].mean(axis=0)
c1 = Xpc[lab == 1, :2].mean(axis=0)
plt.scatter([c0[0], c1[0]], [c0[1], c1[1]], marker='X', s=140, edgecolor='k', linewidths=1)

plt.xlabel(f"PC1 ({vr[0]*100:.1f}% var)")
plt.ylabel(f"PC2 ({vr[1]*100:.1f}% var)" if n_pc >= 2 else "PC2")
plt.title(f"k=2 on PCs from concat({Ksel}ch × {L}t); N={N}; window{WINDOW}")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


ei_c0 = snips[:,:,lab == 0].mean(axis=2).astype(np.float32)   # [K, L]
ei_c1 = snips[:,:,lab == 1].mean(axis=2).astype(np.float32)   # [K, L]
base_cos, base_nch, base_lag, base_olen = cosine_two_eis(
    ei_c0, ei_c1, rms_gate=5.0, use_abs=True, best_align_lag=3
)

fig = plt.figure(figsize=(25, 10))
ax = fig.add_subplot(111)
pew.plot_ei_waveforms(
    [ei_c0, ei_c1], positions=ei_positions, scale=70.0, ax=ax, colors=["black", "red"], 
    alpha=1.0, linewidth=1.0, box_height=1.0, box_width=50.0, aspect=0.5,
)
ax.set_title(f"EIS, cosine: {base_cos}")
plt.show()


# fig = plt.figure(figsize=(25, 10))
# ax = fig.add_subplot(111)
# pew.plot_ei_waveforms(
#     [ei_c0-ei_c1], positions=ei_positions, scale=70.0, ax=ax, colors=["black", "red"], 
#     alpha=1.0, linewidth=1.0, box_height=1.0, box_width=50.0, aspect=0.5,
# )
# ax.set_title(f"EIS, cosine: {base_cos}")
# plt.show()

# 9) handy outputs
idx0 = np.flatnonzero(lab == 0)
idx1 = np.flatnonzero(lab == 1)
times0 = valid_times[idx0]
times1 = valid_times[idx1]
print({"N_total": int(n_total), "N_used": int(N), "n0": int(idx0.size), "n1": int(idx1.size),
       "Ksel": int(Ksel), "L": int(L), "RMS_THRESH": float(RMS_THRESH)})




In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from axolotl_utils_ram import extract_snippets_fast_ram

# -------- params you can tweak --------
win = (-40, 80)            # use the SAME window everywhere
SUBSAMPLE_MAX = 5_000
RMS_THRESH = 5.0           # µV, EI RMS gate for selecting channels
EV_FLAG = 5.0              # %EV threshold to flag a channel
N_PC = 10                  # we only read PC1/PC2 EV, but compute a few PCs
RNG = np.random.default_rng(123)
# --------------------------------------

ev_hits = []               # channels where max(PC1, PC2) >= EV_FLAG
ev_metrics = []            # metrics for all accepted channels

for ch in [4, 14, 24, 63, 93, 95, 161, 234, 253, 399]:#ange(512):
    # 1) valley-based cohort for this channel
    try:
        step1 = find_valley_and_times(
            raw_data, ch,
            window=win, start=0, stop=None,
            bin_width=10.0, valley_bins=5,
            min_valid_count=500,
            ratio_base=3, ratio_step=500, ratio_floor=2, ratio_cap=10
        )
        assert step1['accepted'], "valley acceptance failed"
    except Exception as e:
        # silently skip channels that don't pass the valley check
        # print(f"[skip ch {ch}] {e}")  # uncomment for debugging
        continue

    left_times = np.asarray(step1['left_times'], dtype=np.int64)
    if left_times.size == 0:
        continue

    # uniform subsample to cap cost
    if left_times.size > SUBSAMPLE_MAX:
        left_times = left_times[RNG.choice(left_times.size, SUBSAMPLE_MAX, replace=False)]

    # 2) extract ALL channels (snips: [K, L, N])
    snips, valid_times = extract_snippets_fast_ram(
        raw_data, left_times, window=win, selected_channels=None
    )
    K, L, N = snips.shape
    if N < 50:   # too flimsy for PCA
        continue

    # 3) EI mean and RMS gate for significant channels
    ei_mean = snips.mean(axis=2).astype(np.float32)     # [K, L]
    rms = np.sqrt(np.mean(ei_mean**2, axis=1))          # [K]
    sel = np.flatnonzero(rms > RMS_THRESH)
    if sel.size == 0:
        # fallback: keep top-16 by RMS so we can still compute a signal
        sel = np.argsort(rms)[-16:]
        sel.sort()

    Ksel = sel.size

    # 4) per-spike feature = raw concatenation of selected channels (no z-score/DC)
    X = snips[sel, :, :].transpose(2, 0, 1).reshape(N, Ksel * L).astype(np.float32)
    n_pc = int(min(N_PC, X.shape[0], X.shape[1]))
    if n_pc < 2:
        continue

    # 5) PCA; read EV% for PC1/PC2
    pca = PCA(n_components=n_pc, svd_solver='randomized', random_state=0)
    pca.fit(X)
    vr = pca.explained_variance_ratio_
    ev1 = float(vr[0] * 100.0)
    ev2 = float(vr[1] * 100.0)
    evmax = max(ev1, ev2)   

    Xpc = pca.fit_transform(X)       # [N, n_pc]
    km = KMeans(n_clusters=2, n_init=10, random_state=0)
    lab = km.fit_predict(Xpc)        # [N]

    ei_c0 = snips[:,:,lab == 0].mean(axis=2).astype(np.float32)   # [K, L]
    ei_c1 = snips[:,:,lab == 1].mean(axis=2).astype(np.float32)   # [K, L]
    base_cos, base_nch, base_lag, base_olen = cosine_two_eis(
        ei_c0, ei_c1, rms_gate=5.0, use_abs=True, best_align_lag=3
    )

    rec = {"ch": ch, "N": int(N), "Ksel": int(Ksel), "EV1": ev1, "EV2": ev2, "EVmax": evmax, "Cosine": base_cos}
    ev_metrics.append(rec)



    from sklearn.neighbors import NearestNeighbors

    # assumes you already have:
    # Xpc : [N, ≥2] PCA features (we'll use PC1–PC2)
    # lab : [N] cluster labels in {0,1}

    P2 = Xpc[:, :2].astype(np.float32)
    labels = lab.astype(int)

    idxA = np.flatnonzero(labels == 0)
    idxB = np.flatnonzero(labels == 1)
    A = P2[idxA]
    B = P2[idxB]
    N0, N1 = A.shape[0], B.shape[0]

    if N0 < 2 or N1 < 2:
        print({"N0": int(N0), "N1": int(N1), "msg": "too few points in one cluster for NN test"})
    else:
        # within-cluster nearest (exclude self via n_neighbors=2 → take the 2nd)
        nnA_self = NearestNeighbors(n_neighbors=2, algorithm='auto').fit(A)
        dA_self = nnA_self.kneighbors(A, n_neighbors=2, return_distance=True)[0][:, 1]

        nnB_self = NearestNeighbors(n_neighbors=2, algorithm='auto').fit(B)
        dB_self = nnB_self.kneighbors(B, n_neighbors=2, return_distance=True)[0][:, 1]

        # cross-cluster nearest (just the closest in the other cluster)
        nnA_to_B = NearestNeighbors(n_neighbors=1, algorithm='auto').fit(B)
        dA_cross = nnA_to_B.kneighbors(A, n_neighbors=1, return_distance=True)[0][:, 0]

        nnB_to_A = NearestNeighbors(n_neighbors=1, algorithm='auto').fit(A)
        dB_cross = nnB_to_A.kneighbors(B, n_neighbors=1, return_distance=True)[0][:, 0]

        leakA = dA_cross < dA_self   # points in A whose nearest B is closer than nearest A
        leakB = dB_cross < dB_self   # points in B whose nearest A is closer than nearest B

        out = {
            "N0": int(N0), "N1": int(N1),
            "leakA_count": int(leakA.sum()), "leakA_frac": float(leakA.mean()),
            "leakB_count": int(leakB.sum()), "leakB_frac": float(leakB.mean()),
            "leak_total": int(leakA.sum() + leakB.sum()),
            "leak_frac_total": float((leakA.sum() + leakB.sum()) / (N0 + N1)),
        }
        print(out)

        # (optional) indices you can overlay if you want to visualize later
        leak_global_idx = np.concatenate([idxA[leakA], idxB[leakB]])




    if evmax >= EV_FLAG:
        ev_hits.append(rec)
        print(f"[HIT] ch {ch:3d}  EVmax={evmax:.2f}%  Cosine={base_cos:.2f}")

# ---- summary for inspection ----
ev_hits_sorted = sorted(ev_hits, key=lambda r: r["EVmax"], reverse=True)
print(f"\nTotal hits ≥{EV_FLAG:.1f}%: {len(ev_hits_sorted)} / 512\nTop suspects:")
for r in ev_hits_sorted[:30]:
    print(f"ch {r['ch']:3d}  EVmax={r['EVmax']:.2f}%")

suspect_channels = [r["ch"] for r in ev_hits_sorted]  # <- handy list for follow-up plotting


In [ ]:
ev_metrics

In [ ]:
import numpy as np

# -------- params you can tweak --------
win = (-40, 80)              # SAME window as elsewhere
TAU = 2.0                    # amp threshold = TAU × median(|left_vals|)
FRAC_FLAG = 0.01             # flag if ≥1% of spikes exceed threshold
COUNT_FLAG = 50              # and at least this many spikes exceed threshold
MIN_N = 500                  # guard (matches your min_valid_count)
# --------------------------------------

amp_hits = []
amp_metrics = []

for ch in range(0,512):
    print(ch)
    # build step1 only (fast; no snippet extraction)
    try:
        step1 = find_valley_and_times(
            raw_data, ch,
            window=win, start=0, stop=None,
            bin_width=10.0, valley_bins=5,
            min_valid_count=MIN_N,
            ratio_base=3, ratio_step=500, ratio_floor=2, ratio_cap=10
        )
        assert step1['accepted'], "valley acceptance failed"
    except Exception:
        continue

    vals = np.asarray(step1['left_vals'], dtype=np.float64)  # trough values (likely negative)
    N = vals.size
    if N < MIN_N:
        continue

    amp = np.abs(vals)                      # work in µV magnitude
    med = float(np.median(amp))
    if med <= 0:
        continue

    thr = TAU * med
    count_hi = int(np.sum(amp > thr))
    frac_hi = float(count_hi / N)

    rec = {
        "ch": ch, "N": int(N),
        "median_amp": med,
        "threshold_amp": float(thr),
        "count_hi": count_hi,
        "frac_hi": frac_hi,
        "p95": float(np.percentile(amp, 95)),
        "p99": float(np.percentile(amp, 99)),
        "max": float(amp.max())
    }
    amp_metrics.append(rec)

    if (frac_hi >= FRAC_FLAG) and (count_hi >= COUNT_FLAG):
        amp_hits.append(rec)
    print(f"[AMP HIT] ch {ch:3d}  N={N}  med={med:.1f}µV  thr={thr:.1f}µV  "
            f"count_hi={count_hi}  frac_hi={100*frac_hi:.2f}%  "
            f"p99={rec['p99']:.1f}  max={rec['max']:.1f}")

# ---- summary ----
amp_hits_sorted = sorted(amp_hits, key=lambda r: r["frac_hi"], reverse=True)
print(f"\nTotal amp hits (tau={TAU}, frac≥{FRAC_FLAG}, count≥{COUNT_FLAG}): {len(amp_hits_sorted)} / 512")
for r in amp_hits_sorted[:30]:
    print(f"ch {r['ch']:3d}  frac_hi={100*r['frac_hi']:.2f}%  N={r['N']}  "
          f"med={r['median_amp']:.1f}  thr={r['threshold_amp']:.1f}  "
          f"p99={r['p99']:.1f}  max={r['max']:.1f}")

suspect_channels_by_amp = [r["ch"] for r in amp_hits_sorted]


In [ ]:
# RAW-DATA LIGHTHOUSE PROBE (single channel)
# ------------------------------------------
# Scans one channel for local minima (no refractory):
#    x[i-1] > x[i] and x[i+1] >= x[i]
# Limits to first `max_samples` or until the histogram looks representative.
# Plots:
#   1) Histogram of minima values (even 50-µV bins, y-axis capped)
#   2) Overlay of first 500 waveforms
#   3) Overlay of 500 deepest waveforms
#
# Example call is at the bottom.

import numpy as np
import matplotlib.pyplot as plt
from axolotl_utils_ram import extract_snippets_fast_ram  # uses (T_total, C) int16 array

def scan_and_plot_channel_minima(
    raw_data,            # ndarray (T_total, C), int16
    ch,                  # int, channel index to probe
    *,
    max_samples=1_000_000,     # scan only the first N samples of this channel
    window=(-40, 80),          # snippet window around each detected peak
    first_k=500,               # number of earliest peaks to plot
    top_k=500,                 # number of deepest peaks to plot
    bin_width=50.0,            # µV bin width for histogram
    y_max=500                  # y-axis cap for histogram
):
    """
    Scan one channel for local minima (no refractory), plot histogram and overlay waveforms.
    """
    T_total, C = raw_data.shape
    if not (0 <= ch < C):
        raise ValueError(f"Channel {ch} out of range 0..{C-1}")

    # --- 1) Slice the channel and (optionally) limit to first max_samples
    n = min(int(max_samples), T_total)
    x = raw_data[:n, ch].astype(np.float32)

    # --- 2) Find local minima: x[i-1] > x[i] and x[i+1] >= x[i]
    if n < 3:
        print(f"[ch {ch}] not enough samples to detect minima.")
        return

    left = x[1:-1] < x[:-2]
    right = x[1:-1] <= x[2:]
    minima_mask = left & right
    peak_idx = np.where(minima_mask)[0] + 1   # shift by 1 due to 1..n-2 indexing

    if peak_idx.size == 0:
        print(f"[ch {ch}] no local minima found in first {n} samples.")
        return

    # --- 3) Peak amplitudes (negative values at minima)
    peak_vals = x[peak_idx]                   # negative numbers

    # --- 4) Histogram of peak values (even 50-µV bins, displayed high→low)
    vmin = float(np.nanmin(peak_vals))
    vmax = float(np.nanmax(peak_vals))

    # Align edges to the 50-µV grid (or whatever bin_width you pass)
    low_edge  = bin_width * np.floor(vmin / bin_width)
    high_edge = bin_width * np.ceil (vmax / bin_width)
    if high_edge == low_edge:
        high_edge += bin_width  # guard for degenerate case

    edges = np.arange(low_edge, high_edge + bin_width, bin_width)  # **increasing** edges

    plt.figure(figsize=(6, 3))
    plt.hist(peak_vals, bins=edges, color='gray', edgecolor='black', alpha=0.85)
    plt.title(f"Channel {ch} — local minima (n={peak_idx.size}, first {n} samples)")
    plt.xlabel("value at local minimum (µV)"); plt.ylabel("count")
    plt.ylim(0, y_max)

    # Show from largest (least negative) to smallest (most negative)
    plt.xlim(high_edge, low_edge)  # flips the x-axis without breaking the bin rule
    plt.tight_layout(); plt.show()


    # --- 5) Extract snippets on this channel only
    # Select earliest 'first_k' peaks (chronological)
    first_sel = peak_idx[:min(first_k, peak_idx.size)].astype(np.int64)

    # Select 'top_k' deepest peaks (most negative values)
    if top_k > 0:
        order_top = np.argsort(peak_vals)[:min(top_k, peak_vals.size)]   # ascending => most negative first
        top_sel = peak_idx[order_top].astype(np.int64)
    else:
        top_sel = np.array([], dtype=np.int64)

    # Helper to extract and plot bundles
    def _extract_and_plot(peaks_sel, title_suffix):
        if peaks_sel.size == 0:
            print(f"[ch {ch}] {title_suffix}: no peaks selected.")
            return

        snips, valid_times = extract_snippets_fast_ram(
            raw_data=raw_data,
            spike_times=peaks_sel,
            window=window,
            selected_channels=np.array([ch], dtype=np.int32)
        )
        if snips.shape[2] == 0:
            print(f"[ch {ch}] {title_suffix}: all selected peaks near edges; no valid snippets.")
            return

        wfs = snips[0]           # [L, N]
        L = wfs.shape[0]
        t = np.arange(L)

        plt.figure(figsize=(8, 4))
        plt.plot(t, wfs, color='k', alpha=0.10, linewidth=0.5)
        plt.plot(t, np.median(wfs, axis=1), color='C1', linewidth=2.0, label='median')
        plt.title(f"Channel {ch} — {title_suffix} (N={wfs.shape[1]})")
        plt.xlabel("sample"); plt.ylabel("µV"); plt.grid(True, alpha=0.2)
        plt.legend(fontsize=9)
        plt.tight_layout(); plt.show()

    # _extract_and_plot(first_sel, f"first {first_sel.size} peaks")
    # _extract_and_plot(top_sel,  f"top {top_sel.size} peaks by depth")

    q = np.quantile(peak_vals, [0.1, 0.5, 0.9])
    print(f"[ch {ch}] minima found: {peak_idx.size} (in first {n} samples). "
          f"Peak value quantiles (µV): q10={q[0]:.1f}, q50={q[1]:.1f}, q90={q[2]:.1f}")

scan_and_plot_channel_minima(
        raw_data,
        ch=4,
        max_samples=72_000_000,
        window=(-40, 80),
        first_k=1000,
        top_k=1000,
        bin_width=10.0,
        y_max=700
    )

#### plots

In [ ]:
# EI

fig = plt.figure(figsize=(25, 10))
ax = fig.add_subplot(111)
pew.plot_ei_waveforms(
    step4['final_ei'], positions=ei_positions, scale=70.0, ax=ax, colors=["black", "red"], 
    alpha=1.0, linewidth=1.0, box_height=1.0, box_width=50.0, aspect=0.5,
)
ax.set_title('Final EI')
plt.show()

spikes_in_samples = np.sort(step4['final_times'])  # ensure ascending

# STA and time course
sta0, red_tc0, green_tc0, blue_tc0 = compute_and_plot_sta(spikes_in_samples, triggers_sec, STA_DEPTH=30, STA_OFFSET=2, STA_CHUNK=1000, STA_REFRESH=2, SEED=22222, W=20, H=40, label="all", peak_frame=None, mode="bw")

# --- Parameters ---
sample_rate_hz = 20_000   # 20 kHz
bin_width_ms   = 0.5
max_ms         = 300.0
dt_ms          = 1000
sigma_ms       = 2500

# --- ISI histogram ---
t_ms = (spikes_in_samples / float(sample_rate_hz)) * 1000.0  # Spikes in milliseconds
isi_ms = np.diff(t_ms)
bins = np.arange(0.0, max_ms + bin_width_ms, bin_width_ms)
counts, edges = np.histogram(isi_ms, bins=bins)
centers = edges[:-1] + 0.5 * bin_width_ms

# # --- ISI without right spikes---
# t_ms = (step1['left_times'] / float(sample_rate_hz)) * 1000.0  # Spikes in milliseconds
# isi_ms = np.diff(t_ms)
# bins = np.arange(0.0, max_ms + bin_width_ms, bin_width_ms)
# counts_left, edges = np.histogram(isi_ms, bins=bins)
# centers_left = edges[:-1] + 0.5 * bin_width_ms


# --- Smoothed firing rate ---
dt = dt_ms / 1000
sigma_samples = sigma_ms / dt_ms
spikes_sec = (spikes_in_samples / float(sample_rate_hz))
total_duration = spikes_sec.max() + 0.1 if len(spikes_sec) > 0 else 1.0
time_vector = np.arange(0, total_duration, dt)
rate_counts, _ = np.histogram(spikes_sec, bins=np.append(time_vector, total_duration))
rate = gaussian_filter1d(rate_counts / dt, sigma=sigma_samples)


# --- Combined figure ---
fig, axes = plt.subplots(1, 2, figsize=(20, 3))

# Left: ISI histogram
axes[0].plot(centers, counts, lw=1.5)
# axes[0].plot(centers_left, counts_left, lw=0.5)
axes[0].set_xlim(0, max_ms)
axes[0].set_xlabel("ISI (ms)")
axes[0].set_ylabel("Count")
axes[0].set_title(f"ISI histogram (bin={bin_width_ms} ms, max={max_ms} ms)\nN pairs={isi_ms.size}, Unit 1")

# Right: Smoothed firing rate
axes[1].plot(time_vector, rate, color='red')
axes[1].set_xlim(0, total_duration)
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Hz")
axes[1].set_title("Smoothed Firing Rate")

plt.tight_layout()
plt.show()


# high/low spikes 
amps        = step4['final_amplitudes']   # positive trough magnitudes on main channel
window_ms = 50.0 # ms

# Compute features (no plotting):
m = compute_prev_metrics(step4['final_times'], sample_rate_hz=sample_rate_hz, window_ms=window_ms)
print("Median prev-ISI (ms):", np.nanmedian(m['isi_prev_ms']),
      " | median prev-count:", np.median(m['prev_count']))

# Plot both scatter panels:
plot_amp_vs_prev_metrics(
    step4['final_times'], amps,
    sample_rate_hz=sample_rate_hz,
    window_ms=window_ms,
    max_points=150000,        # downsample if you have huge N
    alpha=0.15,
    title_suffix=f"ch {ch}, window={window_ms} ms"
)

plot_amp_prev_scatter_and_bar(
    step4['final_times'], amps,
    sample_rate_hz=sample_rate_hz,
    window_ms=window_ms,
    max_k=20,            # show k = 0..20 (20+ collapsed into the last bar)
    min_count=5,        # only show bars with ≥20 spikes
    error='std',
    alpha=0.15,
    title_suffix=f"ch {ch}, window={window_ms} ms"
);

In [ ]:
from joint_utils import cosine_two_eis   
from collision_utils import median_ei_adaptive   

rk_vals = step1['rightk_times_by_amp']
right_spikes = np.sort(rk_vals[:100])  # ensure ascending

C = raw_data.shape[1]
all_ch = np.arange(C, dtype=np.int32)

snips_right, right_valid = extract_snippets_fast_ram(
        raw_data=raw_data,  spike_times=right_spikes,  window=(-40,80),  selected_channels=all_ch
    )
ei_right  = median_ei_adaptive(snips_right)
base_cos, base_nch, base_lag, base_olen = cosine_two_eis(
    step3['ref_ei'], ei_right, rms_gate=5.0, use_abs=True, best_align_lag=3)
print(f"Cosine: {base_cos:.2f}, ch: {base_nch}")



In [ ]:

from joint_utils import cosine_two_eis   
from collision_utils import median_ei_adaptive   

rk_vals = step1['rightk_times_by_amp']
right_spikes = np.sort(rk_vals[:100])  # ensure ascending
sta0 = compute_and_plot_sta(right_spikes, triggers_sec, STA_DEPTH=30, STA_OFFSET=0, STA_CHUNK=1000, STA_REFRESH=2, SEED=33333, W=20, H=40, label="top 100", peak_frame=None)

C = raw_data.shape[1]
all_ch = np.arange(C, dtype=np.int32)

snips_right, right_valid = extract_snippets_fast_ram(
        raw_data=raw_data,  spike_times=right_spikes,  window=(-40,80),  selected_channels=all_ch
    )
ei_right  = median_ei_adaptive(snips_right)
base_cos, base_nch, base_lag, base_olen = cosine_two_eis(
    step3['ref_ei'], ei_right, rms_gate=5.0, use_abs=True, best_align_lag=3)
print(f"Cosine: {base_cos:.2f}, ch: {base_nch}")


right_spikes = np.sort(rk_vals[100:200])  # ensure ascending
sta0 = compute_and_plot_sta(right_spikes, triggers_sec, STA_DEPTH=30, STA_OFFSET=0, STA_CHUNK=1000, STA_REFRESH=2, SEED=33333, W=20, H=40, label="second 100", peak_frame=None)

right_spikes = np.sort(rk_vals[200:300])  # ensure ascending
sta0 = compute_and_plot_sta(right_spikes, triggers_sec, STA_DEPTH=30, STA_OFFSET=0, STA_CHUNK=1000, STA_REFRESH=2, SEED=33333, W=20, H=40, label="third 100", peak_frame=None)

right_spikes = np.sort(rk_vals[300:400])  # ensure ascending
sta0 = compute_and_plot_sta(right_spikes, triggers_sec, STA_DEPTH=30, STA_OFFSET=0, STA_CHUNK=1000, STA_REFRESH=2, SEED=33333, W=20, H=40, label="forth 100", peak_frame=None)

right_spikes = np.sort(rk_vals[400:500])  # ensure ascending
sta0 = compute_and_plot_sta(right_spikes, triggers_sec, STA_DEPTH=30, STA_OFFSET=0, STA_CHUNK=1000, STA_REFRESH=2, SEED=33333, W=20, H=40, label="fifth 100", peak_frame=None)


In [ ]:

# assumes you already have:
#   left_times : np.ndarray[int64]  # absolute sample indices (from find_valley_and_times)
#   left_vals  : np.ndarray[float]  # same length; negative trough values (more negative = larger amplitude)
#   triggers_sec : np.ndarray[float]
#   compute_and_plot_sta(...) is in scope

def split_left_into_amplitude_bins(
    left_times: np.ndarray,
    left_vals:  np.ndarray,
    *,
    bin_size: int = 1000,
    deepest_first: bool = True,
    drop_last: bool = False
):
    """
    Split LEFT spikes into fixed-size bins by amplitude.
    Returns a list of dicts with 'times', 'vals', and simple metadata.

    left_vals are assumed negative (more negative = larger amplitude).
    If deepest_first=True, the first bin is the most negative (largest amplitude) spikes.
    """
    assert left_times.shape[0] == left_vals.shape[0], "left_times/left_vals length mismatch"
    n = left_times.shape[0]
    if n == 0:
        return []

    # Sort by amplitude (negative trough -> more negative is deeper)
    order = np.argsort(left_vals)              # ascending: most negative first
    if not deepest_first:
        order = order[::-1]
    t_sorted = left_times[order]
    v_sorted = left_vals[order]

    # Number of bins
    if drop_last:
        n_bins = n // bin_size
    else:
        n_bins = (n + bin_size - 1) // bin_size

    bins = []
    for b in range(n_bins):
        start = b * bin_size
        end   = min((b + 1) * bin_size, n)
        if start >= end:
            break
        seg_t = t_sorted[start:end]
        seg_v = v_sorted[start:end]
        bins.append({
            "idx": b,
            "times": seg_t,                    # absolute sample indices (int64)
            "vals":  seg_v.astype(np.float32),
            "n":     int(end - start),
            "amp_min": float(seg_v.min()),     # most negative (largest magnitude) in this bin
            "amp_max": float(seg_v.max()),     # least negative in this bin
        })
    return bins

# -----------------------
# Example usage / loop:
# -----------------------
# 1) Make 1k-spike amplitude bins (deepest first)
bins = split_left_into_amplitude_bins(
    left_times=step1['left_times'],
    left_vals=step1['left_vals'],
    bin_size=1000,
    deepest_first=True,   # first bin = largest amplitude (most negative)
    drop_last=False            # keep a possibly shorter last bin
)

print(f"Total LEFT spikes: {len(step1['left_times'])}, bins: {len(bins)}")

# # 2) Run your STA per bin (you can add more plots here)
# for b in bins:
#     label = f"left_amp_bin {b['idx']+1}  n={b['n']}  amp[{b['amp_min']:.1f}..{b['amp_max']:.1f}] µV"
#     spikes = b["times"]  # absolute sample indices
#     print(f"Computing STA for {label}")
#     sta0 = compute_and_plot_sta(spikes, triggers_sec, STA_DEPTH=30, STA_OFFSET=2, STA_CHUNK=1000, STA_REFRESH=2, SEED=22222, W=20, H=40, label="all", peak_frame=None, mode="bw")
#     # You can add more per-bin plots/analysis here, using `spikes` and `b["vals"]`.

green_traces = []
eis = []
if not bins:
    print("No bins to process.")
else:
    picks = [0, len(bins) // 2, len(bins) - 2]
    picks = sorted(set(i for i in picks if 0 <= i < len(bins)))  # dedupe & bounds-check
    # picks = [0,7,10]

    for j in picks:
        b = bins[j]
        label = f"left_amp_bin {b['idx']+1}  n={b['n']}  amp[{b['amp_min']:.1f}..{b['amp_max']:.1f}] µV"
        spikes = b["times"]  # absolute sample indices
        print(f"Computing STA for {label}")
        sta, red_tc, green_tc, blue_tc = compute_and_plot_sta(spikes, triggers_sec, STA_DEPTH=30, STA_OFFSET=2, STA_CHUNK=1000, STA_REFRESH=2, SEED=22222, W=20, H=40, label="all", peak_frame=None, mode="bw")
        green_traces.append(green_tc)  # accumulate for later plotting

        sn_pool, t_pool_valid = extract_snippets_fast_ram(
            raw_data=raw_data,  # (T,C)
            spike_times=spikes,
            window=(-40,80),
            selected_channels=np.arange(512, dtype=np.int32),
        )
        ei = sn_pool.mean(axis=2)
        eis.append(ei)



    plt.figure(figsize=(10, 4))
    for i, trace in enumerate(green_traces):
        plt.plot(trace, label=f"bin {i+1}", lw=1.5)
    plt.plot(green_tc0, color="black", label="All", lw=2)
    plt.title("Green timecourse per amplitude bin")
    plt.xlabel("Time (frames)")
    plt.ylabel("Green channel response")
    plt.legend()
    plt.tight_layout()
    plt.show()

#     fig = plt.figure(figsize=(25, 10))
#     ax = fig.add_subplot(111)
#     pew.plot_ei_waveforms(
#         eis, positions=ei_positions, scale=70.0, ax=ax, colors=["black", "red", "green"], 
#         alpha=1.0, linewidth=1.0, box_height=1.0, box_width=50.0, aspect=0.5,
#     )
#     ax.set_title('Final EI')
#     plt.show()


In [ ]:
print(picks)

In [ ]:
fig = plt.figure(figsize=(25, 15))
ax = fig.add_subplot(111)
pew.plot_ei_waveforms(
    eis, positions=ei_positions, scale=70.0, ax=ax, colors=["black", "red", "green"], 
    alpha=1.0, linewidth=1.0, box_height=1.0, box_width=50.0, aspect=0.9,
)
ax.set_title('Final EI')
plt.show()


In [ ]:
plt.figure(figsize=(10, 4))
for i, trace in enumerate(green_traces):
    plt.plot(trace, label=f"bin {i+1}", lw=1.5)
plt.plot(green_tc0, color="black", label="All", lw=2)
plt.title("Green timecourse per amplitude bin")
plt.xlabel("Time (frames)")
plt.ylabel("Green channel response")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:

# high/low spikes 
amps        = step4['final_amplitudes']   # positive trough magnitudes on main channel
window_ms = 30.0 # ms

# Compute features (no plotting):
m = compute_prev_metrics(step4['final_times'], sample_rate_hz=sample_rate_hz, window_ms=window_ms)
print("Median prev-ISI (ms):", np.nanmedian(m['isi_prev_ms']),
      " | median prev-count:", np.median(m['prev_count']))

# Plot both scatter panels:
plot_amp_vs_prev_metrics(
    step4['final_times'], amps,
    sample_rate_hz=sample_rate_hz,
    window_ms=window_ms,
    max_points=150000,        # downsample if you have huge N
    alpha=0.15,
    title_suffix=f"ch {ch}, window={window_ms} ms"
);

plot_amp_prev_scatter_and_bar(
    step4['final_times'], amps,
    sample_rate_hz=sample_rate_hz,
    window_ms=window_ms,
    max_k=20,            # show k = 0..20 (20+ collapsed into the last bar)
    min_count=5,        # only show bars with ≥20 spikes
    error='std',
    alpha=0.15,
    title_suffix=f"ch {ch}, window={window_ms} ms"
);

# STA

In [ ]:
import numpy as np

# Collapse to absolute to find max deviation
sta_abs = np.abs(sta0)
idx_flat = np.argmin(sta0)  # most negative pixel
x_peak, y_peak, c_peak, t_peak = np.unravel_index(idx_flat, sta0.shape)

print(f"Darkest pixel at (x={x_peak}, y={y_peak}, channel={c_peak}, time={t_peak})")
green_tc = sta0[x_peak, y_peak, 1, :]  # Green channel only
# Normalize green_tc for projection
template = green_tc / np.linalg.norm(green_tc)

# Compute projection for each pixel and color channel
projection = np.tensordot(sta0, template, axes=([3], [0]))  # shape: [W, H, C]
proj_2d = projection.mean(axis=2)  # shape: [W, H]
mean_val = np.median(proj_2d)
robust_std = np.median(np.abs(proj_2d - mean_val)) * 1.4826  # MAD-to-STD

threshold = mean_val + 8 * robust_std
rf_mask = proj_2d > threshold  # shape: [W, H]
rf_pixels = np.argwhere(rf_mask)  # list of (x, y) coords

print(f"Identified {len(rf_pixels)} RF pixels")
rf_tcs = []
ratios = []

for (x, y) in rf_pixels:
    tc = sta0[x, y, :, :]  # shape: [3, T]
    rf_tcs.append(tc)

    peak_idx = np.argmin(tc.sum(axis=0))  # time point of strongest response
    rgb_at_peak = tc[:, peak_idx]  # [R, G, B] at peak
    
    # Normalize to green = 1
    green_val = rgb_at_peak[1] if rgb_at_peak[1] != 0 else 1e-6
    ratio = rgb_at_peak / green_val
    ratios.append(ratio)

rf_tcs = np.stack(rf_tcs)        # [N_pixels, 3, T]
ratios = np.stack(ratios)        # [N_pixels, 3]
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.imshow(proj_2d.T, origin='lower', cmap='hot')  # transpose to match display convention
plt.colorbar(label="STA projection (arbitrary units)")
for (x, y) in rf_pixels:
    plt.plot(x, y, 'g*', markersize=8)
plt.title("2D STA projection with RF pixels")
plt.xlabel("X (width)"); plt.ylabel("Y (height)")
plt.tight_layout()
plt.show()
T = sta0.shape[3]
time = np.arange(T)

plt.figure(figsize=(10, len(rf_pixels) * 1.5))
for i, tc in enumerate(rf_tcs):
    plt.subplot(len(rf_pixels), 1, i+1)
    plt.plot(time, tc[0], 'r', label='R')
    plt.plot(time, tc[1], 'g', label='G')
    plt.plot(time, tc[2], 'b', label='B')
    plt.axvline(np.argmin(tc.sum(axis=0)), color='k', linestyle='--')
    plt.title(f"Pixel {i} (x={rf_pixels[i][0]}, y={rf_pixels[i][1]})")
    if i == 0:
        plt.legend()
plt.tight_layout()
plt.show()
ind = np.arange(len(rf_pixels))
width = 0.3

plt.figure(figsize=(10, 4))
plt.bar(ind - width, ratios[:, 0], width, label='R', color='r')
plt.bar(ind,         ratios[:, 1], width, label='G', color='g')
plt.bar(ind + width, ratios[:, 2], width, label='B', color='b')
plt.axhline(1.0, color='gray', linestyle='--')
plt.xlabel("RF pixel index")
plt.ylabel("Ratio (normalized to G = 1)")
plt.legend()
plt.title("RGB weights per RF pixel (at STA peak)")
plt.tight_layout()
plt.show()


In [ ]:
from extract_bw_snippets import extract_bw_snippets

try:
    # preferred in your repo
    from benchmark_c_rgb_generation import RGBFrameGenerator
except ImportError:
    # fallback if you keep the class in compute_sta_from_spikes.py
    from compute_sta_from_spikes import RGBFrameGenerator


In [ ]:
import numpy as np

def classify_spike_groups(spike_times,
                          window_ms=50.0,
                          in_samples=False,
                          fs=None,
                          return_indices=False):
    """
    Bucket 'first spikes' by how many additional spikes follow within window_ms.
    A spike is a 'first spike' iff there are NO spikes in the preceding window_ms.

    Args:
        spike_times: 1D array of spike times (seconds by default; or samples if in_samples=True)
        window_ms: window size in milliseconds (before/after)
        in_samples: if True, spike_times are in samples
        fs: sampling rate (Hz). Required if in_samples=True
        return_indices: also return indices of the first spikes (into the sorted spike_times)

    Returns:
        groups_times: dict {k: np.array of first-spike times (sec)} where k is # of spikes in the next window
        groups_idx (optional): dict {k: np.array of indices into the sorted spike_times}
    """
    st = np.asarray(spike_times, dtype=float)
    if in_samples:
        if fs is None:
            raise ValueError("fs is required when in_samples=True")
        st = st / fs
    st = np.sort(st)
    N = len(st)

    W = window_ms / 1000.0
    groups_times = {}
    groups_idx = {} if return_indices else None

    # Two pointers for previous-window (l) and next-window (r)
    l = 0
    r = 0
    for i in range(N):
        # advance l so that st[i] - st[l] <= W OR l == i
        while l < i and (st[i] - st[l]) > W:
            l += 1
        prev_any = (i - l) > 0
        if prev_any:
            continue  # not a first spike, skip

        # advance r to first index with st[r] - st[i] > W
        if r < i + 1:
            r = i + 1
        while r < N and (st[r] - st[i]) <= W:
            r += 1

        k_after = r - (i + 1)  # number of spikes in (i, i+W]
        groups_times.setdefault(k_after, []).append(st[i])
        if return_indices:
            groups_idx.setdefault(k_after, []).append(i)

    groups_times = {k: np.array(v) for k, v in groups_times.items()}
    if return_indices:
        groups_idx = {k: np.array(v, dtype=int) for k, v in groups_idx.items()}
        return groups_times, groups_idx
    return groups_times


In [ ]:
import numpy as np
from axolotl_utils_ram import extract_snippets_fast_ram
from collision_utils import median_ei_adaptive
from typing import Optional

def build_train_position_eis(
    raw_data: np.ndarray,
    spikes_in_samples: np.ndarray,   # 1D int64, chronological, absolute sample indices
    group_first_indices: np.ndarray, # 1D int (indices into spikes_in_samples) for a fixed group (e.g., groups_idx[k])
    k: int,                          # number of EXTRA spikes after the first in the train (so train length = k+1)
    *,
    window: tuple = (-40, 80),
    selected_channels: Optional[np.ndarray] = None,  # None → all channels
    reducer: str = "median",         # 'median' (median_ei_adaptive) or 'mean'
) -> dict:
    """
    For a given group of mini-trains (all of the same length k+1), build average EIs for
    each spike POSITION j = 0..k across trains.

    - POSITION 0 = the first spike of each train (indices = group_first_indices + 0)
    - POSITION 1 = the second spike of each train (indices = group_first_indices + 1)
    - ...
    - POSITION k = the (k+1)-th spike of each train

    Returns:
      {
        'eis_by_order': list[(C,L)] of length (k+1),  # EI for each position
        'times_by_order': list[np.ndarray[int64]],     # valid times used per position (after edge filtering)
        'counts_by_order': list[int],                  # N per position after extraction validity
        'selected_channels': np.ndarray[int32],        # channels used
        'window': tuple,
      }
    """
    spikes_in_samples = np.asarray(spikes_in_samples, dtype=np.int64)
    group_first_indices = np.asarray(group_first_indices, dtype=np.int64)

    if group_first_indices.size == 0:
        raise ValueError("group_first_indices is empty")

    # Basic safety: make sure i + k stays in-bounds for all i (should be true if groups were built correctly)
    max_needed = int(np.max(group_first_indices) + k)
    if max_needed >= spikes_in_samples.size:
        # Drop any offending starters near the end
        ok = (group_first_indices + k) < spikes_in_samples.size
        group_first_indices = group_first_indices[ok]
        if group_first_indices.size == 0:
            raise RuntimeError("All starts in group are too close to the end for k extra spikes.")

    # Channel set
    C_total = int(raw_data.shape[1])
    if selected_channels is None:
        selected_channels = np.arange(C_total, dtype=np.int32)
    else:
        selected_channels = np.asarray(selected_channels, dtype=np.int32)

    # For each position j, gather absolute times and extract snippets
    eis_by_order = []
    times_by_order = []
    counts_by_order = []

    for j in range(k + 1):
        idx_j = group_first_indices + j
        times_j = spikes_in_samples[idx_j].astype(np.int64, copy=False)

        # Extract snippets on selected channels only
        sn_j, valid_times_j = extract_snippets_fast_ram(
            raw_data=raw_data,
            spike_times=times_j,
            window=window,
            selected_channels=selected_channels
        )  # sn_j: (C_sel, L, N_j_valid)

        sn_j = sn_j.astype(np.float32, copy=False)
        if sn_j.shape[2] == 0:
            # Keep shape-consistency with an empty placeholder (rare, usually near edges)
            raise RuntimeError(f"No valid snippets for train position j={j}; check window/edges.")

        # Reduce to EI for this position
        if reducer == "median":
            ei_j = median_ei_adaptive(sn_j)      # (C_sel, L)
        else:
            ei_j = sn_j.mean(axis=2)

        eis_by_order.append(ei_j)
        times_by_order.append(valid_times_j.astype(np.int64, copy=False))
        counts_by_order.append(int(sn_j.shape[2]))

    return {
        "eis_by_order": eis_by_order,
        "times_by_order": times_by_order,
        "counts_by_order": counts_by_order,
        "selected_channels": selected_channels,
        "window": tuple(window),
    }

import matplotlib.pyplot as plt
from plot_ei_waveforms import plot_ei_waveforms

def plot_train_position_overlay(
    eis_by_order: list,
    ei_positions: np.ndarray,
    *,
    ref_channel: Optional[int] = None,
    title: Optional[str] = None,

    scale: float = 70.0,
    box_height: float = 1.0,
    box_width: float = 50.0,
):
    plt.figure(figsize=(20, 12))
    # plot_ei_waveforms can accept a list of EIs to overlay
    plot_ei_waveforms(
        eis_by_order,
        ei_positions=ei_positions,
        scale=scale,
        box_height=box_height,
        box_width=box_width,
        ref_channel=ref_channel
    )
    if title:
        plt.suptitle(title)
    plt.show()

# Suppose you have: raw_data, spikes_in_samples, groups_idx (e.g., a dict), ei_positions
k = 3
res = build_train_position_eis(
    raw_data, spikes_in_samples, first_indices_by_k[k], k=k,
    window=(-40, 80), selected_channels=np.array([166], dtype=np.int32), reducer='median'
)

# Assume `res` is from build_train_position_eis(...)
eis = res['eis_by_order']          # list of (C_sel, L)
ch_local = 0                       # with one extracted channel, this is 0
pre = -res['window'][0]            # time-zero index
L = eis[0].shape[1]
x = np.arange(L) - pre             # samples; to get ms: x = x / 20000.0 * 1000.0

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 5))
for j, ei in enumerate(eis):
    y = ei[ch_local]               # shape (L,)
    plt.plot(x, y, label=f"#{j+1}")
plt.axvline(0, lw=0.8, alpha=0.6)
plt.xlabel("Samples")              # or "Time (ms)" if you convert x
plt.ylabel("Amplitude")
plt.title("Average spikes by train position (single channel)")
plt.legend(title="Position", ncol=2, frameon=False)
plt.tight_layout()
plt.show()


In [ ]:
# seconds
groups, groups_idx = classify_spike_groups(np.asarray(spikes_in_samples, dtype=float) / FS, window_ms=50, return_indices=True)

# or, if you have samples:
# groups = classify_spike_groups(spike_times_samples, window_ms=50, in_samples=True, fs=20000)

# e.g., how many singletons / doubles / triples:
counts = {k: len(v) for k, v in groups.items()}
print(counts)

# get the list of first-spike times for “singletons” and “doubles”:
singletons = groups.get(0, np.array([]))
doubles    = groups.get(1, np.array([]))


In [ ]:
isi = np.diff(np.sort(np.asarray(spikes_in_samples, dtype=float) / FS * 1000))
print("Min ISI:", np.min(isi))
print("Median ISI:", np.median(isi))
print("Max ISI:", np.max(isi))


In [ ]:
import numpy as np

def spikes_outside_first_windows(spikes_sec, group_firsts, window_ms=50.0):
    """
    Return spikes that are NOT covered by any [first, first+W] window.
    (These are exactly the ones you're noticing: in long trains, later spikes
    can fall >50 ms after the first, yet still have short ISIs.)
    """
    st = np.asarray(spikes_sec, dtype=float)
    st.sort()
    if len(group_firsts) == 0:
        return st.copy()

    s = np.asarray(group_firsts, dtype=float)
    s.sort()
    W = window_ms / 1000.0

    # For each first spike, mark the index range [start_idx, end_idx)
    start_idx = np.searchsorted(st, s, side='left')
    end_idx   = np.searchsorted(st, s + W, side='right')  # inclusive window

    # Prefix-sum trick to mark covered indices without looping over all spikes
    covered = np.zeros(st.shape[0] + 1, dtype=int)
    np.add.at(covered, start_idx, 1)
    np.add.at(covered, end_idx,  -1)
    covered = np.cumsum(covered)[:-1] > 0  # boolean: spike covered by some first-window?

    return st[~covered], covered  # outside, mask


group_firsts = np.concatenate([v for v in groups.values()]) if len(groups) else np.array([])
outside_spikes, covered_mask = spikes_outside_first_windows(np.asarray(spikes_in_samples, dtype=float) / FS, group_firsts, 50)



In [ ]:
print(len(outside_spikes))

In [ ]:
import numpy as np

def segment_adaptive_trains(
    spike_times,
    in_samples=False,
    fs=None,
    alpha=3.0,                 # multiplies the robust min-ISI to get the first allowed gap
    cap_ms=20.0,               # saturation cap for allowed ISI
    cap_at_gap_index=7,        # cap reached by the 7th gap (i.e., between 7th and 8th spike)
    min_isi_percentile=0.5,    # robust "minimum" ISI (e.g., 0.5th–1st percentile)
    floor_ms=0.5               # don't let base be crazy-small (units: ms)
):
    """
    Partition spikes into adaptive trains with a relaxing gap threshold.

    A train grows while the next ISI <= allowed(gap_index), where:
      allowed(1) = T0 = alpha * robust_min_ISI
      allowed(k) increases linearly up to cap_ms by gap k = cap_at_gap_index
      allowed(k >= cap_at_gap_index) = cap_ms

    Returns:
      st           : sorted spike times (seconds)
      trains       : list of (start_idx, end_idx) half-open index ranges into `st`
      train_id     : array of length N mapping each spike to its train id
      pos_in_train : array of length N giving 1-based position of a spike within its train
      sched_info   : dict with schedule parameters actually used (T0, cap_ms, etc.)
    """
    st = np.asarray(spike_times, dtype=float)
    if in_samples:
        if fs is None:
            raise ValueError("fs is required when in_samples=True")
        st = st / fs
    st = np.sort(st)
    N = st.size
    if N == 0:
        return st, [], np.array([], dtype=int), np.array([], dtype=int), {}

    # Robust "min" ISI (ms)
    isis_ms = np.diff(st) * 1000.0
    if isis_ms.size == 0:
        # Only one spike: it's one singleton train
        trains = [(0, 1)]
        return st, trains, np.array([0]), np.array([1]), {
            "T0_ms": None, "cap_ms": cap_ms, "alpha": alpha,
            "cap_at_gap_index": cap_at_gap_index, "robust_min_isi_ms": None
        }

    robust_min = max(np.percentile(isis_ms, min_isi_percentile), floor_ms)
    T0 = alpha * robust_min

    # Precompute a small schedule array for gap indices 1..cap_at_gap_index
    # Linear ramp from T0 to cap_ms
    if cap_at_gap_index < 1:
        cap_at_gap_index = 1
    if cap_at_gap_index == 1:
        sched = np.array([min(T0, cap_ms)], dtype=float)
    else:
        ks = np.arange(1, cap_at_gap_index + 1, dtype=float)
        frac = (ks - 1) / (cap_at_gap_index - 1)  # 0..1
        sched = T0 + (cap_ms - T0) * frac
        sched = np.minimum(sched, cap_ms)

    def allowed_gap_ms(gap_idx: int) -> float:
        """gap_idx is 1 for (spike1→spike2), 2 for (spike2→spike3), ..."""
        if gap_idx <= cap_at_gap_index:
            return sched[gap_idx - 1]
        return cap_ms

    trains = []
    train_id = -np.ones(N, dtype=int)
    pos_in_train = np.zeros(N, dtype=int)

    i = 0
    tid = 0
    while i < N:
        start = i
        pos = 1
        i += 1
        gap_idx = 1  # next gap to check will be between spikes at indices (i-1) and i

        while i < N:
            isi_ms_here = (st[i] - st[i - 1]) * 1000.0
            thr = allowed_gap_ms(gap_idx)
            if isi_ms_here <= thr:
                i += 1
                gap_idx += 1
            else:
                break

        end = i  # exclusive
        trains.append((start, end))
        train_id[start:end] = tid
        # positions 1..len(train)
        pos_in_train[start:end] = np.arange(1, end - start + 1, dtype=int)
        tid += 1

    sched_info = {
        "T0_ms": float(T0),
        "cap_ms": float(cap_ms),
        "alpha": float(alpha),
        "cap_at_gap_index": int(cap_at_gap_index),
        "robust_min_isi_ms": float(robust_min)
    }
    return st, trains, train_id, pos_in_train, sched_info


st, trains, train_id, pos_in_train, info = segment_adaptive_trains(np.asarray(spikes_in_samples, dtype=float) / FS, alpha=6.0, cap_ms = 20.0)

# First-spike times (one per train) and sizes
first_times = np.array([st[s] for s, e in trains])
sizes       = np.array([e - s for s, e in trains])  # spikes per train

# Buckets by “# following spikes”
first_by_followers = {}
for sz in np.unique(sizes):
    k_follow = sz - 1
    first_by_followers[k_follow] = first_times[sizes == sz]

print(info)  # see T0, cap, etc.

# Compute sizes of each train
sizes = np.array([end - start for start, end in trains])  # number of spikes per train

# Get the first spike time in each train
first_spike_times = np.array([st[start] for start, end in trains])

# Bucket by number of *following* spikes (i.e., train size - 1)
groups = {}
for k_follow in np.unique(sizes - 1):
    groups[k_follow] = first_spike_times[sizes - 1 == k_follow]

# Example: how many singletons, doubles, etc.
for k, v in sorted(groups.items()):
    print(f"{k+1}-spike trains: {len(v)} first spikes")

sizes = np.array([end - start for (start, end) in trains])
first_indices_by_k = {}
first_indices = np.array([start for (start, end) in trains])  # shape: (N_trains,)

for k_follow in np.unique(sizes - 1):
    mask = (sizes - 1) == k_follow
    first_indices_by_k[k_follow] = first_indices[mask]


In [ ]:

W = 20
H = 40
FS = 20_000.0                 # spike sampling rate (Hz)
LIB_PATH = "/Volumes/Lab/Users/alexth/axolotl/sta/libdraw_rgb.so"

lut = np.array([              # 8-color LUT 
    [255, 255, 255],
    [255, 255,   0],
    [255,   0, 255],
    [255,   0,   0],
    [0,   255, 255],
    [0,   255,   0],
    [0,     0, 255],
    [0,     0,   0]
], dtype=np.uint8).flatten()

# --- Configure generator  ---
gen = RGBFrameGenerator(LIB_PATH)
gen.configure(width=W, height=H, lut=lut, noise_type=1, n_bits=3)

spikes_sec = groups[0]

rf_pixels_tr = [ (y, x) for (x, y) in rf_pixels ]  # swap x/y

stim_snips = extract_bw_snippets(
    spikes_sec=spikes_sec,
    triggers_sec=triggers_sec,
    generator=gen,
    seed=33333,
    rf_pixels=rf_pixels_tr,       # list of (x, y)
    weights_rgb=ratios.mean(axis=0),   # or average across RF if you want
    depth=20,
    offset=5,
    chunk_size=1000,
    refresh=2
)

print(stim_snips.shape)  # [N_spikes, 15, N_pixels]

# Average over spikes
mean_snips = stim_snips.mean(axis=0)  # shape: [15, 4]

spikes_sec = groups[2]

rf_pixels_tr = [ (y, x) for (x, y) in rf_pixels ]  # swap x/y

stim_snips = extract_bw_snippets(
    spikes_sec=spikes_sec,
    triggers_sec=triggers_sec,
    generator=gen,
    seed=33333,
    rf_pixels=rf_pixels_tr,       # list of (x, y)
    weights_rgb=ratios.mean(axis=0),   # or average across RF if you want
    depth=20,
    offset=5,
    chunk_size=1000,
    refresh=2
)

print(stim_snips.shape)  # [N_spikes, 15, N_pixels]

# Average over spikes
mean_snips1 = stim_snips.mean(axis=0)  # shape: [15, 4]


spikes_sec = groups[4]

rf_pixels_tr = [ (y, x) for (x, y) in rf_pixels ]  # swap x/y

stim_snips = extract_bw_snippets(
    spikes_sec=spikes_sec,
    triggers_sec=triggers_sec,
    generator=gen,
    seed=33333,
    rf_pixels=rf_pixels_tr,       # list of (x, y)
    weights_rgb=ratios.mean(axis=0),   # or average across RF if you want
    depth=20,
    offset=5,
    chunk_size=1000,
    refresh=2
)

print(stim_snips.shape)  # [N_spikes, 15, N_pixels]

# Average over spikes
mean_snipsx = stim_snips.mean(axis=0)  # shape: [15, 4]



spikes_sec = np.asarray(spikes_in_samples, dtype=float) / FS

stim_snips = extract_bw_snippets(
    spikes_sec=spikes_sec,
    triggers_sec=triggers_sec,
    generator=gen,
    seed=33333,
    rf_pixels=rf_pixels_tr,       # list of (x, y)
    weights_rgb=ratios.mean(axis=0),   # or average across RF if you want
    depth=20,
    offset=5,
    chunk_size=1000,
    refresh=2
)

print(stim_snips.shape)  # [N_spikes, 15, N_pixels]

# Average over spikes
mean_snips0 = stim_snips.mean(axis=0)  # shape: [15, 4]

# Plot each pixel’s time course
plt.figure(figsize=(8, 5))
# for i in range(mean_snips.shape[1]):
#     plt.plot(mean_snips[:, i], label=f'Pixel {i}', color='red')
#     plt.plot(mean_snips1[:, i], label=f'Pixel {i}', color='blue')
#     plt.plot(mean_snips0[:, i], label=f'Pixel {i}', color='black')
plt.plot(mean_snips.mean(axis=1),  color='red',   label='Singletons')
plt.plot(mean_snips1.mean(axis=1), color='blue',  label='4 spikes')
plt.plot(mean_snips0.mean(axis=1), color='black', label='All')
plt.plot(mean_snipsx.mean(axis=1), color='green', label='7 spikes')

plt.xlabel("Frames before spike (0 = most recent)")
plt.ylabel("BW Stimulus Intensity")
plt.title("Mean stimulus history preceding spikes (per pixel)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Average over spikes
mean_snips = stim_snips.mean(axis=0)  # shape: [15, 4]

# Plot each pixel’s time course
plt.figure(figsize=(8, 5))
for i in range(mean_snips.shape[1]):
    plt.plot(mean_snips[:, i], label=f'Pixel {i}')

plt.xlabel("Frames before spike (0 = most recent)")
plt.ylabel("BW Stimulus Intensity")
plt.title("Mean stimulus history preceding spikes (per pixel)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Pick random examples
n_examples = 6
example_indices = np.random.choice(stim_snips.shape[0], size=n_examples, replace=False)

plt.figure(figsize=(10, n_examples * 2))
for i, idx in enumerate(example_indices):
    snippet = stim_snips[idx]  # shape: [15, 4]
    plt.subplot(n_examples, 1, i+1)
    for pix in range(snippet.shape[1]):
        plt.plot(snippet[:, pix], label=f'Pixel {pix}')
    plt.title(f"Stimulus snippet {idx}")
    if i == 0:
        plt.legend()
    plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Frame 12 (0-indexed) is the 13th frame before the spike
target_frame = 12

# Find where all 4 pixels were negative at that frame
mask = np.all(stim_snips[:, target_frame, :] < 0, axis=1)  # shape: [N_spikes]

# Filter those snippets
dark_snips = stim_snips[mask]  # shape: [N_dark, 15, 4]

print(f"Selected {dark_snips.shape[0]} snippets with all 4 pixels < 0 at frame {target_frame}")

# Compute mean
mean_dark = dark_snips.mean(axis=0)  # shape: [15, 4]

# Plot
plt.figure(figsize=(8, 5))
for i in range(mean_dark.shape[1]):
    plt.plot(mean_dark[:, i], label=f'Pixel {i}')
plt.axvline(target_frame, color='k', linestyle='--', label='Selection frame')
plt.xlabel("Frame (0 = most recent)")
plt.ylabel("BW Stimulus Intensity")
plt.title("Mean stimulus history for 'dark-at-frame-12' subset")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Frame 12 (0-indexed) is the 13th frame before the spike
target_frame = 9

# Find where all 4 pixels were negative at that frame
mask = np.all(stim_snips[:, target_frame, :] > 0, axis=1)  # shape: [N_spikes]

# Filter those snippets
dark_snips = stim_snips[mask]  # shape: [N_dark, 15, 4]

print(f"Selected {dark_snips.shape[0]} snippets with all 4 pixels < 0 at frame {target_frame}")

# Compute mean
mean_dark = dark_snips.mean(axis=0)  # shape: [15, 4]

# Plot
plt.figure(figsize=(8, 5))
for i in range(mean_dark.shape[1]):
    plt.plot(mean_dark[:, i], label=f'Pixel {i}')
plt.axvline(target_frame, color='k', linestyle='--', label='Selection frame')
plt.xlabel("Frame (0 = most recent)")
plt.ylabel("BW Stimulus Intensity")
plt.title("Mean stimulus history for 'dark-at-frame-12' subset")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
frame_a = 8
frame_b = 9
frame_c = 7

# Select where all 4 pixels were > 0 at both frames
mask_pos = np.all(stim_snips[:, frame_a, :] > 0, axis=1) & np.all(stim_snips[:, frame_b, :] > 0, axis=1) & np.all(stim_snips[:, frame_c, :] > 0, axis=1)


# Filter snippets
bright_snips = stim_snips[mask_pos]
print(f"Selected {bright_snips.shape[0]} snippets with all pixels > 0 at frames {frame_a} and {frame_b}")

# Compute mean
mean_bright = bright_snips.mean(axis=0)  # shape: [15, 4]

# Plot
plt.figure(figsize=(8, 5))
for i in range(mean_bright.shape[1]):
    plt.plot(mean_bright[:, i], label=f'Pixel {i}')
plt.axvline(frame_a, color='k', linestyle='--', label='Bright frames')
plt.axvline(frame_b, color='k', linestyle='--')
plt.xlabel("Frame (0 = most recent)")
plt.ylabel("BW Stimulus Intensity")
plt.title("Mean stimulus history for 'bright-at-frames-9+10' subset")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
frame_a = 10
frame_b = 12

# Select where all 4 pixels were > 0 at both frames
mask_pos = np.all(stim_snips[:, frame_a, :] > 0, axis=1) & np.all(stim_snips[:, frame_b, :] < 0, axis=1)

# Filter snippets
bright_snips = stim_snips[mask_pos]
print(f"Selected {bright_snips.shape[0]} snippets with all pixels > 0 at frames {frame_a} and {frame_b}")

# Compute mean
mean_bright = bright_snips.mean(axis=0)  # shape: [15, 4]

# Plot
plt.figure(figsize=(8, 5))
for i in range(mean_bright.shape[1]):
    plt.plot(mean_bright[:, i], label=f'Pixel {i}')
plt.axvline(frame_a, color='k', linestyle='--', label='Bright frames')
plt.axvline(frame_b, color='k', linestyle='--')
plt.xlabel("Frame (0 = most recent)")
plt.ylabel("BW Stimulus Intensity")
plt.title("Mean stimulus history for 'bright-at-frames-9+10' subset")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Reshape to [N_spikes, features]
X = dark_snips.reshape(dark_snips.shape[0], -1)  # shape: [72850, 60]

# Run PCA
pca = PCA(n_components=10)
X_pca = pca.fit_transform(X)  # shape: [72850, 10]

# Plot PC1 vs PC2
plt.figure(figsize=(6, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], s=2, alpha=0.4, c='black')
plt.xlabel('PC 1')
plt.ylabel('PC 2')
plt.title('Spike-triggered stimulus space (PC1 vs PC2)')
plt.grid(True)
plt.tight_layout()
plt.show()

# Optional: variance explained
print("Variance explained:", pca.explained_variance_ratio_[:5])


# mess with manual

#### extract valley and left spikes

In [ ]:
# VALLEY → BUILD A 500-SPIKE “RIGHT” SET (VALLEY + FILL FROM FURTHER RIGHT)
# ------------------------------------------------------------------------
# For one channel:
#   1) find local minima (no refractory)
#   2) find a negative-side “valley” window of `valley_bins` bins that satisfies:
#        • valley_count ≤ valley_max_count
#        • left_count   > 2 * valley_count
#        • left_count   > min_valid_count
#   3) LEFT  = spikes strictly more negative than the valley (vals < valley_low)
#   4) RIGHT500 = ALL spikes inside the valley (vals ∈ [valley_low, valley_high)),
#                 plus the most-negative spikes to the right of the valley
#                 (vals ≥ valley_high) to reach a total of `right_k` (default 500).
#                 If valley has ≥ right_k spikes, keep the right_k most negative from valley.
#
# Extract ALL-CHANNEL snippets for LEFT and RIGHT500.
# No plots; returns times/values both chronological and amplitude-sorted.

import numpy as np
from axolotl_utils_ram import extract_snippets_fast_ram

def _local_minima_no_refractory(xf: np.ndarray) -> np.ndarray:
    """Indices i where xf[i-1] > xf[i] and xf[i+1] >= xf[i]."""
    n = xf.size
    if n < 3:
        return np.empty(0, dtype=np.int64)
    left  = xf[1:-1] <  xf[:-2]
    right = xf[1:-1] <= xf[ 2:]
    return (np.where(left & right)[0] + 1).astype(np.int64)

def detect_valley_and_extract_snips_rightk(
    raw_data: np.ndarray,   # (T_total, C) int16/float
    ch: int,                # channel to analyze (only used for minima detection)
    *,
    max_samples: int = 1_000_000,
    bin_width: float = 10.0,      # ADC units
    valley_bins: int = 5,         # consecutive bins in the valley
    valley_max_count: int = 50,   # max total counts in the valley window
    min_valid_count: int = 300,   # minimum required left_count
    window: tuple = (-40, 80),    # (pre, post)
    right_k: int = 500            # total spikes “to the right” (valley + beyond)
):
    """
    Returns
    -------
    out : dict
        'channel'                : int
        'n_minima'               : int
        'valley_low_edge'        : float
        'valley_high_edge'       : float
        'valley_count'           : int
        'left_count'             : int

        # LEFT (vals < valley_low)
        'left_times_sorted'      : (N_left,) int64  (chronological)
        'left_vals_sorted'       : (N_left,) float
        'left_times_by_amp'      : (N_left,) int64  (most negative first)
        'left_vals_by_amp'       : (N_left,) float
        'snips_left'             : (C, L, N_left_valid) float32
        'times_left_valid'       : (N_left_valid,) int64

        # RIGHT500 = all valley spikes + fill from right side to reach right_k
        'right500_times_sorted'  : (N_r500,) int64  (chronological of selected set)
        'right500_vals_sorted'   : (N_r500,) float
        'right500_times_by_amp'  : (N_r500,) int64  (most negative first, within selected set)
        'right500_vals_by_amp'   : (N_r500,) float
        'snips_right500'         : (C, L, N_r500_valid) float32
        'times_right500_valid'   : (N_r500_valid,) int64
    Raises
    ------
    RuntimeError if no acceptable valley is found (per the stricter rule).
    """
    T_total, C = raw_data.shape
    if not (0 <= ch < C):
        raise ValueError(f"Channel {ch} out of range 0..{C-1}")

    # 1) channel slice & minima
    n = min(int(max_samples), T_total)
    x = raw_data[:n, ch].astype(np.float32, copy=False)

    idx_all  = _local_minima_no_refractory(x)
    vals_all = x[idx_all]
    if idx_all.size == 0:
        raise RuntimeError(f"[ch {ch}] no local minima in first {n} samples.")

    # 2) negative-side histogram to find valley
    vmin = float(np.nanmin(vals_all))
    if vmin >= 0:
        raise RuntimeError(f"[ch {ch}] no negative minima — cannot define valley.")

    bin_width = float(bin_width)
    low_edge  = bin_width * np.floor(vmin / bin_width)
    high_edge = 0.0
    edges = np.arange(low_edge, high_edge + bin_width, bin_width, dtype=np.float64)
    if edges.size < valley_bins + 1:
        low_edge = low_edge - bin_width * ((valley_bins + 1) - edges.size + 1)
        edges = np.arange(low_edge, high_edge + bin_width, bin_width, dtype=np.float64)

    counts, edges = np.histogram(vals_all, bins=edges)
    nb = counts.size

    found = False
    for j in range(0, nb - valley_bins + 1):
        # window [edges[j], edges[j+valley_bins]) must lie entirely below 0
        if edges[j + valley_bins] > 0.0:
            continue

        valley_cnt = int(np.sum(counts[j:j + valley_bins]))
        if valley_cnt > valley_max_count:
            continue

        left_cnt = int(np.sum(counts[:j])) if j > 0 else 0

        # *** Stricter acceptance: left > 2*valley  AND  left > min_valid_count
        if (left_cnt > 2 * valley_cnt) and (left_cnt > int(min_valid_count)):
            valley_low  = float(edges[j])               # more negative edge
            valley_high = float(edges[j + valley_bins]) # closer to 0
            found = True
            print(f"Found a valley with valley count {valley_cnt} and left count {left_cnt}")
            break

    if not found:
        raise RuntimeError(
            f"[ch {ch}] no acceptable valley window found "
            f"(need left > 2*valley and left > {min_valid_count})."
        )

    # 3) masks & sorted orders
    left_mask   = (vals_all <  valley_low)
    valley_mask = (vals_all >= valley_low) & (vals_all < valley_high)
    right_mask  = (vals_all >= valley_high)

    # Chronological order
    order_time  = np.argsort(idx_all)
    idx_sorted  = idx_all[order_time]
    vals_sorted = vals_all[order_time]

    left_times_sorted   = idx_sorted[left_mask[order_time]]
    left_vals_sorted    = vals_sorted[left_mask[order_time]]
    valley_times_sorted = idx_sorted[valley_mask[order_time]]
    valley_vals_sorted  = vals_sorted[valley_mask[order_time]]
    right_times_sorted  = idx_sorted[right_mask[order_time]]
    right_vals_sorted   = vals_sorted[right_mask[order_time]]

    # Amplitude order (ascending values ⇒ more negative first)
    left_order_by_amp   = np.argsort(left_vals_sorted)
    val_order_by_amp    = np.argsort(valley_vals_sorted)
    right_order_by_amp  = np.argsort(right_vals_sorted)

    left_times_by_amp   = left_times_sorted[left_order_by_amp]
    left_vals_by_amp    = left_vals_sorted[left_order_by_amp]
    valley_times_by_amp = valley_times_sorted[val_order_by_amp]
    valley_vals_by_amp  = valley_vals_sorted[val_order_by_amp]
    right_times_by_amp  = right_times_sorted[right_order_by_amp]
    right_vals_by_amp   = right_vals_sorted[right_order_by_amp]

    # 4) Build RIGHT500 as: all valley spikes, plus fill from right-of-valley to reach right_k
    #    Prioritize most-negative within each pool.
    target_k = int(right_k)
    vN = valley_times_by_amp.size

    if vN >= target_k:
        # More than enough in valley: keep the target_k most negative from valley
        r500_times = valley_times_by_amp[:target_k]
        r500_vals  = valley_vals_by_amp[:target_k]
    else:
        needed = target_k - vN
        # Take most-negative `needed` from right-of-valley
        fill_times = right_times_by_amp[:min(needed, right_times_by_amp.size)]
        fill_vals  = right_vals_by_amp[:min(needed, right_vals_by_amp.size)]
        r500_times = np.concatenate([valley_times_by_amp, fill_times]).astype(np.int64, copy=False)
        r500_vals  = np.concatenate([valley_vals_by_amp,  fill_vals]).astype(np.float32, copy=False)

    # Chronological & amplitude views for the selected RIGHT500
    r500_order_time = np.argsort(r500_times)
    right500_times_sorted = r500_times[r500_order_time]
    right500_vals_sorted  = r500_vals[r500_order_time]

    r500_order_amp = np.argsort(r500_vals)  # ascending ⇒ more negative first
    right500_times_by_amp = r500_times[r500_order_amp]
    right500_vals_by_amp  = r500_vals[r500_order_amp]

    # 5) Extract ALL-CHANNEL snippets
    pre, post = int(window[0]), int(window[1])
    all_ch = np.arange(C, dtype=np.int32)

    def _extract(times):
        if times.size == 0:
            L = post - pre + 1
            return np.empty((C, L, 0), dtype=np.float32), np.array([], dtype=np.int64)
        sn, valid = extract_snippets_fast_ram(
            raw_data=raw_data, spike_times=times.astype(np.int64),
            window=(pre, post), selected_channels=all_ch
        )
        return sn.astype(np.float32, copy=False), valid.astype(np.int64, copy=False)

    snips_left,   valid_left    = _extract(left_times_sorted)
    snips_r500,   valid_r500    = _extract(right500_times_sorted)

    # 6) Package output
    out = {
        'channel': int(ch),
        'n_minima': int(idx_all.size),
        'valley_low_edge':  float(valley_low),
        'valley_high_edge': float(valley_high),
        'valley_count': int(np.sum(valley_mask)),
        'left_count':   int(np.sum(left_mask)),

        'left_times_sorted': left_times_sorted.astype(np.int64,  copy=False),
        'left_vals_sorted':  left_vals_sorted.astype(np.float32, copy=False),
        'left_times_by_amp': left_times_by_amp.astype(np.int64,  copy=False),
        'left_vals_by_amp':  left_vals_by_amp.astype(np.float32, copy=False),
        'snips_left':        snips_left,
        'times_left_valid':  valid_left,

        'right500_times_sorted': right500_times_sorted.astype(np.int64,  copy=False),
        'right500_vals_sorted':  right500_vals_sorted.astype(np.float32, copy=False),
        'right500_times_by_amp': right500_times_by_amp.astype(np.int64,  copy=False),
        'right500_vals_by_amp':  right500_vals_by_amp.astype(np.float32, copy=False),
        'snips_right500':        snips_r500,
        'times_right500_valid':  valid_r500,
    }
    return out


# -----------------------------
# Example call (single channel)
# -----------------------------
ch = 350
res = detect_valley_and_extract_snips_rightk(
    raw_data,
    ch=ch,
    max_samples=72_000_000,
    bin_width=10.0,
    valley_bins=5,
    valley_max_count=893,
    min_valid_count=300,   # stricter left requirement
    window=(-40, 80),
    right_k=500
)
print(f"ch {res['channel']} | valley [{res['valley_low_edge']:.1f},{res['valley_high_edge']:.1f}) | "
      f"left={res['left_times_sorted'].size} right500={res['right500_times_sorted'].size}")
print("snips_left   :", res['snips_left'].shape,   "| snips_right500:", res['snips_right500'].shape)


### build EIs for left_low and left_high

In [ ]:
# EIs FROM LEFT (k = min(500, N_left//2), deepest) AND RIGHT (top 50, with offset)
# ---------------------------------------------------------------------------------
# Adds `first_spike_right`: an offset into the RIGHT amplitude ranking.
#   • first_spike_right = 0   → take the 50 largest (most negative) RIGHT spikes
#   • first_spike_right = 65  → skip the 65 largest, then take the next 50
#
# Assumes `res` comes from detect_valley_and_extract_snips_rightk(...) and contains:
#   - snips_left, times_left_valid, left_times_by_amp
#   - snips_right500, times_right500_valid, right500_times_by_amp

import numpy as np

def build_eis_left_right(
    res,
    *,
    left_cap=500,
    right_cap=50,
    first_spike_right=0,   # skip this many largest RIGHT spikes before taking right_cap
):
    # --- LEFT (deepest k = min(500, N_left_valid//2)) ---
    sn_left   = np.asarray(res['snips_left'],           dtype=np.float32)  # (C, L, N_left_valid)
    t_left_v  = np.asarray(res['times_left_valid'],     dtype=np.int64)    # (N_left_valid,)
    left_amp_times_full = np.asarray(res['left_times_by_amp'], dtype=np.int64)  # most-negative first (pre-valid)

    N_left_valid = sn_left.shape[2]
    k_left = int(min(int(left_cap), N_left_valid // 2))
    if k_left <= 0:
        raise RuntimeError("No valid LEFT spikes to build EI (k_left <= 0).")

    # Map absolute time -> column index in LEFT snippet cube
    left_pos = {int(t): i for i, t in enumerate(t_left_v.tolist())}

    left_idx = []
    left_times_abs = []
    for t in left_amp_times_full:
        j = left_pos.get(int(t), None)
        if j is not None:
            left_idx.append(j)
            left_times_abs.append(int(t))
            if len(left_idx) >= k_left:
                break
    if not left_idx:
        raise RuntimeError("LEFT amplitude-ranked times did not intersect valid-left snippets.")
    left_idx = np.asarray(left_idx, dtype=np.int64)
    left_times_abs = np.asarray(left_times_abs, dtype=np.int64)
    k_left = int(left_idx.size)

    ei_left_k = sn_left[:, :, left_idx].mean(axis=2)  # (C, L)

    # --- RIGHT (top 50, with offset) ---
    sn_right  = np.asarray(res['snips_right500'],        dtype=np.float32)  # (C, L, N_right_valid)
    t_right_v = np.asarray(res['times_right500_valid'],  dtype=np.int64)    # (N_right_valid,)
    right_amp_times_sel = np.asarray(res['right500_times_by_amp'], dtype=np.int64)  # most-negative first (selected pool)

    # Apply offset and cap
    start = int(max(0, first_spike_right))
    if start >= right_amp_times_sel.size:
        raise RuntimeError(f"first_spike_right={start} beyond available RIGHT pool ({right_amp_times_sel.size}).")
    right_amp_times_window = right_amp_times_sel[start:]  # skip `start` most negative

    # Map absolute time -> column index in RIGHT snippet cube
    right_pos = {int(t): i for i, t in enumerate(t_right_v.tolist())}

    k_right_target = int(min(int(right_cap), sn_right.shape[2]))
    right_idx = []
    right_times_abs = []
    for t in right_amp_times_window:
        j = right_pos.get(int(t), None)
        if j is not None:
            right_idx.append(j)
            right_times_abs.append(int(t))
            if len(right_idx) >= k_right_target:
                break

    if not right_idx:
        raise RuntimeError("RIGHT amplitude-ranked times (after offset) did not intersect valid-right snippets.")
    right_idx = np.asarray(right_idx, dtype=np.int64)
    right_times_abs = np.asarray(right_times_abs, dtype=np.int64)
    k_right = int(right_idx.size)

    ei_right = sn_right[:, :, right_idx].mean(axis=2)  # (C, L)

    return {
        'ei_left_k': ei_left_k,                     # (C, L)
        'ei_right_50': ei_right,                    # (C, L)
        'times_left_k': left_times_abs,             # (k_left,)
        'times_right_50': right_times_abs,          # (k_right,)
        'k_left': k_left,
        'k_right': k_right,
        'first_spike_right': start,
    }

# -----------------------------
# Example call
# -----------------------------
out = build_eis_left_right(res, left_cap=500, right_cap=100, first_spike_right=0)
print(out['ei_left_k'].shape, out['ei_right_50'].shape)
print("left_k times (first 5):", out['times_left_k'][:5])
print("right_50 times (first 5):", out['times_right_50'][:5])
#
# # Try skipping the first 65 largest RIGHT spikes:
# # out2 = build_eis_left_right(res, left_cap=500, right_cap=50, first_spike_right=65)


### compare left_low and left_high EIs, plot them, and compute STAs

In [ ]:

sim_cos, sim_nch, sim_lag, sim_olen = cosine_two_eis(
    out['ei_left_k'], out['ei_right_50'],
    rms_gate=5.0,
    use_abs=True,
    best_align_lag=3
)
print(f"[similarity] EIs cos={sim_cos if np.isfinite(sim_cos) else np.nan:.4f} "
    f"(nch={sim_nch}, lag={sim_lag:+d}, overlapL={sim_olen})")

fig = plt.figure(figsize=(20, 12))
ax = fig.add_subplot(111)
pew.plot_ei_waveforms(
    [out['ei_left_k'], out['ei_right_50']],
    positions=ei_positions,
    scale=70.0,
    ax=ax,
    colors=["black", "red"],
    alpha=1.0,
    linewidth=1.0,
    box_height=1.0,
    box_width=50.0,
    aspect=1.0,
)
ax.set_title('Original EI (KS)')
plt.tight_layout()
plt.show()

sp = res['left_times_by_amp']
sta0 = compute_and_plot_sta(out['times_left_k'], triggers_sec, STA_DEPTH=30, STA_OFFSET=0, STA_CHUNK=1000, STA_REFRESH=2, SEED=33333, W=20, H=40, label="high", peak_frame=None)
sta1 = compute_and_plot_sta(out['times_right_50'], triggers_sec, STA_DEPTH=30, STA_OFFSET=0, STA_CHUNK=1000, STA_REFRESH=2, SEED=33333, W=20, H=40, label="Low", peak_frame=None)
# sta2 = compute_and_plot_sta(sp, triggers_sec, label="All")


### sweep valley spikes for cosine with left_low

In [ ]:
# SWEEP FIRST-SPIKE OFFSET ON RIGHT GROUP → COSINE vs OFFSET
# ----------------------------------------------------------
# For each offset s = 0,1,2,...:
#   - Build EI_left_k (deepest k from LEFT) and EI_right_50 (RIGHT group starting at s)
#   - Compute similarity via your cosine_two_eis (with lag search)
#   - Record sim_cos, sim_nch, sim_lag, sim_olen
# Stops when the RIGHT selection cannot produce 50 valid spikes after the offset.
#
# Requires:
#   - `res` from detect_valley_and_extract_snips_rightk(...)
#   - `build_eis_left_right` (with first_spike_right support)
#   - `cosine_two_eis` available in scope

import numpy as np
import matplotlib.pyplot as plt

def sweep_right_offset_cosine(
    res,
    *,
    left_cap=500,
    right_cap=100,
    rms_gate=10.0,
    use_abs=True,
    best_align_lag=3,
    max_offset=None,   # if None, sweep until selection fails
    quiet=False
):
    sim_cos_list = []
    sim_nch_list = []
    sim_lag_list = []
    sim_olen_list = []
    offsets = []

    # If user doesn't set max_offset, use length of amplitude-ranked RIGHT pool as an upper bound
    if max_offset is None:
        max_offset = int(len(res.get('right500_times_by_amp', [])))

    for s in range(max_offset + 1):
        try:
            out = build_eis_left_right(
                res,
                left_cap=left_cap,
                right_cap=right_cap,
                first_spike_right=s
            )
        except RuntimeError as e:
            # Either not enough RIGHT after offset, or no LEFT (should be rare if already validated)
            if not quiet:
                print(f"[offset {s}] stop: {e}")
            break

        ei_left  = out['ei_left_k']
        ei_right = out['ei_right_50']

        # Your function: cosine_two_eis(ei0, ei1, rms_gate, use_abs, best_align_lag)
        sim_cos, sim_nch, sim_lag, sim_olen = cosine_two_eis(
            ei_left, ei_right,
            rms_gate=rms_gate,
            use_abs=use_abs,
            best_align_lag=best_align_lag
        )

        sim_cos_list.append(float(sim_cos))
        sim_nch_list.append(int(sim_nch))
        sim_lag_list.append(int(sim_lag))
        sim_olen_list.append(int(sim_olen))
        offsets.append(s)

        if not quiet and (s % 10 == 0):
            print(f"[offset {s:4d}] cos={sim_cos:.4f} | nch={sim_nch} | lag={sim_lag} | olen={sim_olen}")

    # Convert to arrays
    offsets = np.asarray(offsets, dtype=int)
    sim_cos_arr = np.asarray(sim_cos_list, dtype=float)
    sim_nch_arr = np.asarray(sim_nch_list, dtype=int)
    sim_lag_arr = np.asarray(sim_lag_list, dtype=int)
    sim_olen_arr= np.asarray(sim_olen_list, dtype=int)

    # Plot cosine vs offset (first_spike_right)
    plt.figure(figsize=(8, 3.5))
    plt.plot(offsets, sim_cos_arr, '-o', ms=3, lw=1.5, label='cosine(left_k, right_50@offset)')
    plt.xlabel("first_spike_right offset (skip this many largest RIGHT spikes)")
    plt.ylabel("cosine similarity")
    plt.ylim(0, 1.02)
    plt.grid(True, alpha=0.3)
    plt.title("Cosine vs RIGHT offset")
    plt.legend()
    plt.tight_layout()
    plt.show()

    return {
        "offsets": offsets,
        "sim_cos": sim_cos_arr,
        "sim_nch": sim_nch_arr,
        "sim_lag": sim_lag_arr,
        "sim_olen": sim_olen_arr,
    }

# -----------------------------
# Example call
# -----------------------------
sweep = sweep_right_offset_cosine(
    res,
    left_cap=500,
    right_cap=100,
    rms_gate=5.0,
    use_abs=True,
    best_align_lag=3,
    max_offset=None,   # sweep until the RIGHT pool is exhausted
    quiet=False
)
# The plot shows the trajectory; if you want the raw arrays:
# print(sweep["offsets"][:10], sweep["sim_cos"][:10])
